<div align="center" style="background: linear-gradient(135deg, #0a0a0a 0%, #1a0a1f 100%); padding: 32px 24px; border-radius: 8px; border: 1px solid #3a2555; margin-bottom: 16px;">
  <img src="https://raw.githubusercontent.com/whykusanagi/forza-abyss-painter/main/assets/forza_abyss_painter_logo.png" alt="Forza Abyss Painter" width="180" style="margin-bottom: 16px;"/>
  <h1 style="color: #d94f90; margin: 0; font-family: -apple-system, BlinkMacSystemFont, sans-serif; font-weight: 700; letter-spacing: 0.5px;">Forza Abyss Painter</h1>
  <p style="color: #a78bfa; margin: 8px 0 0 0; font-size: 14px; font-family: -apple-system, BlinkMacSystemFont, monospace;">GPU Shape-Gen · Colab Edition</p>
  <p style="color: #6a6a6a; margin: 4px 0 0 0; font-size: 12px;"><a href="https://github.com/whykusanagi/forza-abyss-painter" style="color: #8b5cf6; text-decoration: none;">github.com/whykusanagi/forza-abyss-painter</a></p>
</div>

This notebook runs the Forza Abyss Painter GPU shape-generator on CUDA.
It mirrors the desktop CLI (`forza_abyss_painter.cli.oneshot`) but is self-contained — no clone, no install
beyond the pre-installed Colab stack.

**Before running:** switch the runtime to a GPU.
> Runtime → Change runtime type → Hardware accelerator: **GPU** (T4 is fine; V100/A100 is faster)

**Workflow:**
1. Run the *Setup* cell (verifies CUDA, defines the engine).
2. Run the *Upload* cell and pick a PNG/JPG. For stickers (RGBA with transparency you want
   preserved), set `STICKER_MODE=True` in the *Run* cell.
3. Run the *Run* cell — it generates shapes and writes `<stem>_<NUM_SHAPES>.json`
   + `<stem>_<NUM_SHAPES>_render.png` so you can tell at a glance which budget tier this
   JSON belongs to (eg `my_image_3000.json` = high-detail vs `my_image_400.json` = lineart).
4. The result is displayed inline. Download the JSON to ship to your Windows FH6 injector.

**If you hit a CUDA OOM:** run the **Cleanup** cell (Section 3) and lower `RANDOM_SAMPLES`
or `MUTATIONS_PER_ROUND`. Memory cost scales as `K * H * W * 12 bytes` per intermediate,
with 2-3 intermediates live. Restart runtime if the cleanup cell can't get allocation under
~10 GB.


## Preset: **MEDIUM / 1000 shapes (multi-shape EVAL — not injectable yet)**

This variant ships tuned defaults for a 1000-shape budget. Edit any knob in the Configure cell.

## 1. Setup — verify CUDA

In [ ]:
# --- Environment check ---
import sys, time
import numpy as np
import torch
from PIL import Image

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:     {name}  ({vram_gb:.1f} GB VRAM)")
else:
    print("WARNING: No CUDA. Runtime > Change runtime type > Hardware accelerator: GPU")

DTYPE = torch.float32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 2. Setup — load the Forza Abyss Painter GPU engine

Run this cell once per session. It defines `run_gpu(...)` and helpers.

In [ ]:
# === Forza Abyss Painter GPU engine — inlined verbatim from forza_abyss_painter/shapegen/gpu/ ===
# Regenerated by notebooks/build_colab_notebook.py. DO NOT edit here; edit the
# package modules and rebuild so the notebook and CLI never drift apart.
from __future__ import annotations
from dataclasses import dataclass, field, replace
from typing import Callable
import time
import numpy as np
import torch


# ----- device.py -----
DTYPE = torch.float32


def get_device() -> torch.device:
    # CUDA first (Colab / discrete GPUs), then Apple MPS, then CPU. This lets the same
    # package code run on a Colab CUDA runtime and on an Apple Silicon Mac unchanged.
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

# ----- rasterize.py -----
def rasterize_rotated_ellipses(params: torch.Tensor, h: int, w: int) -> torch.Tensor:
    """Batched rasterization of K rotated ellipses to a (K, H, W) HARD mask in {0, 1}.

    `params` is (K, 5): columns are (x, y, rx, ry, angle_degrees). Tensor must already live
    on the target device (call `.to(get_device())` before passing in).

    Hard edge (d2 <= 1) at every size — this is the COMMIT/SCORING rasterizer and matches
    both the CPU engine and how FH6 renders vinyl ellipses (crisp vector edges). A soft
    anti-aliased variant (`_ellipse_soft` in shapes_gpu) is used only inside gradient
    refinement, where differentiability is needed. Committing soft edges instead would
    accumulate ~1px fringes across thousands of overlapping translucent shapes into visible
    haze/bleed, and would also overstate smoothness vs. the actual in-game render.
    """
    if params.ndim != 2 or params.shape[1] != 5:
        raise ValueError(f"params must be (K, 5); got {tuple(params.shape)}")
    device = params.device
    K = params.shape[0]
    if K == 0:
        return torch.zeros((0, h, w), dtype=DTYPE, device=device)

    x = params[:, 0].view(K, 1, 1)
    y = params[:, 1].view(K, 1, 1)
    rx = params[:, 2].clamp(min=1e-3).view(K, 1, 1)
    ry = params[:, 3].clamp(min=1e-3).view(K, 1, 1)
    angle_rad = torch.deg2rad(params[:, 4]).view(K, 1, 1)
    cos_a = torch.cos(angle_rad)
    sin_a = torch.sin(angle_rad)

    ys = torch.arange(h, dtype=DTYPE, device=device).view(1, h, 1)
    xs = torch.arange(w, dtype=DTYPE, device=device).view(1, 1, w)

    dx = xs - x   # (K, 1, W)
    dy = ys - y   # (K, H, 1)
    xr = cos_a * dx + sin_a * dy
    yr = -sin_a * dx + cos_a * dy
    nx = xr / rx
    ny = yr / ry
    d2 = nx * nx + ny * ny   # (K, H, W)
    return (d2 <= 1.0).to(DTYPE)

# ----- scoring.py -----
"""Batched scoring for GPU shape-gen prototype.

Memory note: `score_batch` materializes `(K, H, W, 3)` intermediates during the
optimal-color reduction and SSE decomposition. At K=256 and H=W=1200 the peak
unified-memory footprint is roughly 12 GB float32 (rasterizer + scorer combined).
On 8 GB Apple Silicon, reduce K (random_samples / mutations_per_round in GPUConfig)
or max-resolution. 16+ GB unified memory is comfortable for the default settings.
"""



# CPU engine uses alpha=128 for ellipses (see shapes/ellipse.py:72). Lock the same.
ALPHA_FIXED: int = 128

# Fraction of a candidate shape's body that must sit inside the opaque region.
# Re-exports the same constant as forza_abyss_painter/shapegen/scoring.py:99 so the GPU and CPU
# paths reject identically. Imported by the engine and tested for equality with CPU.
STICKER_OVERLAP_MIN: float = 0.995

# A pixel of the rasterized soft mask is part of the "body" if its coverage >= this.
# CPU code uses mask_local >= 128 on a 0..255 mask, which is the same threshold (0.5).
_BODY_THRESHOLD: float = 0.5


def _to_float(t: torch.Tensor) -> torch.Tensor:
    """uint8 HWC → float32 HWC (no normalization; range stays 0..255)."""
    return t.to(DTYPE)


def _score_core(
    masks: torch.Tensor,                       # (K, H, W) float in 0..1
    current_u8: torch.Tensor,                  # (H, W, 3) uint8
    target_u8: torch.Tensor,                   # (H, W, 3) uint8
    alpha_mask: torch.Tensor | None = None,    # (H, W) uint8 or None
    alpha: int = ALPHA_FIXED,
    edge_weight: torch.Tensor | None = None,   # (H, W) float per-pixel loss weight or None
) -> tuple[torch.Tensor, torch.Tensor]:
    """Score K candidate masks at a SINGLE alpha. Returns (scores, colors). See score_batch
    for the public entry point that searches over alpha levels.

    Non-sticker math: optimal src = (target - (1-a)*current) / a averaged over the mask;
                      SSE-after = SSE-before - SSE_inside_old + SSE_inside_new.
    Sticker math: same closed-form but weighted by min(mask, alpha) so transparent pixels
                  don't influence color, and RMS uses (alpha > 0) as the loss weight so
                  transparent pixels are ignored.

    `edge_weight` (H, W), if given, multiplies into the per-pixel loss so high-contrast
    detail (eyes, lineart) counts more — steering the search to spend shapes there. It
    combines multiplicatively with the sticker weight. Optimal color is left as the mask
    average (placement steering is the dominant effect).
    """
    if masks.ndim != 3:
        raise ValueError(f"masks must be (K, H, W); got {tuple(masks.shape)}")
    K, H, W = masks.shape
    device = masks.device

    cur = _to_float(current_u8)   # (H, W, 3)
    tgt = _to_float(target_u8)    # (H, W, 3)
    a = float(alpha) / 255.0
    ew = edge_weight.unsqueeze(-1) if edge_weight is not None else None  # (H, W, 1)

    if alpha_mask is None:
        # ---- Non-sticker (opaque) path ----
        m = masks.unsqueeze(-1)                                    # (K, H, W, 1)
        weight = masks.sum(dim=(1, 2)).clamp(min=1e-3)             # (K,)
        sum_mt = (m * tgt).sum(dim=(1, 2))                         # (K, 3)
        sum_mc = (m * cur).sum(dim=(1, 2))                         # (K, 3)
        src = ((sum_mt - (1.0 - a) * sum_mc) / (a * weight.unsqueeze(-1))).clamp(0.0, 255.0)

        diff_old = cur - tgt                                       # (H, W, 3)
        diff_old_sq = diff_old * diff_old
        src_expanded = src.view(K, 1, 1, 3)
        diff_new_inside = a * src_expanded + (1.0 - a) * cur - tgt
        diff_new_sq = diff_new_inside * diff_new_inside
        # Soft-edge approximation: at fringe pixels (0<m<1) the SSE term overestimates the
        # true post-blend error by a non-negative offset shared by every candidate, so argmin
        # ordering is preserved (CPU reference uses the exact post-blend SSE).
        if ew is None:
            sse_before = diff_old_sq.sum()
            sse_in_old = (m * diff_old_sq).sum(dim=(1, 2, 3))
            sse_in_new = (m * diff_new_sq).sum(dim=(1, 2, 3))
            n = float(H * W * 3)
        else:
            sse_before = (diff_old_sq * ew).sum()
            sse_in_old = (m * diff_old_sq * ew).sum(dim=(1, 2, 3))
            sse_in_new = (m * diff_new_sq * ew).sum(dim=(1, 2, 3))
            n = (ew.sum() * 3.0).clamp(min=1.0)

        sse_after = sse_before - sse_in_old + sse_in_new
        rms = torch.sqrt(sse_after.clamp(min=0.0) / n)
        return rms, src.round().to(torch.uint8)

    # ---- Sticker path ----
    alpha_f = alpha_mask.to(DTYPE) / 255.0                         # (H, W) in 0..1
    opaque_body = (alpha_mask >= 128)                              # (H, W) bool — for overlap check
    weight_full_2d = (alpha_mask > 0).to(DTYPE)                    # (H, W) in {0, 1}
    if edge_weight is not None:
        weight_full_2d = weight_full_2d * edge_weight              # fold in edge emphasis
    weight_full = weight_full_2d.unsqueeze(-1)                     # (H, W, 1) — per-pixel RMS weight

    # Effective mask for optimal-color reduction: AND mask with alpha (in 0..1).
    eff = masks * alpha_f.unsqueeze(0)                             # (K, H, W)
    eff_w = eff.unsqueeze(-1)                                      # (K, H, W, 1)
    eff_weight = eff.sum(dim=(1, 2)).clamp(min=1e-3)               # (K,)

    sum_mt = (eff_w * tgt).sum(dim=(1, 2))                         # (K, 3)
    sum_mc = (eff_w * cur).sum(dim=(1, 2))                         # (K, 3)
    src = ((sum_mt - (1.0 - a) * sum_mc) / (a * eff_weight.unsqueeze(-1))).clamp(0.0, 255.0)

    # Weighted RMS. The shape paints with `masks` (not `eff`); every SSE term is multiplied
    # by `weight_full` (alpha gate × edge emphasis) so transparent pixels never contribute
    # and high-contrast pixels count more.
    m_w = masks.unsqueeze(-1)                                       # (K, H, W, 1)
    diff_old = cur - tgt                                            # (H, W, 3)
    sse_before = ((diff_old * diff_old) * weight_full).sum()        # scalar
    sse_in_old = (m_w * (diff_old * diff_old) * weight_full).sum(dim=(1, 2, 3))
    src_expanded = src.view(K, 1, 1, 3)
    diff_new_inside = a * src_expanded + (1.0 - a) * cur - tgt      # (K, H, W, 3)
    # (same soft-edge approximation as above — argmin ordering is preserved)
    sse_in_new = (m_w * (diff_new_inside * diff_new_inside) * weight_full).sum(dim=(1, 2, 3))

    sse_after = sse_before - sse_in_old + sse_in_new
    n = weight_full_2d.sum() * 3.0
    rms = torch.sqrt(sse_after.clamp(min=0.0) / n.clamp(min=1.0))

    # Sticker overlap constraint: count body pixels and how many sit in opaque area.
    body = masks >= _BODY_THRESHOLD                                 # (K, H, W) bool
    body_total = body.sum(dim=(1, 2)).to(DTYPE)                     # (K,)
    inside = (body & opaque_body.unsqueeze(0)).sum(dim=(1, 2)).to(DTYPE)  # (K,)
    ratio = torch.where(body_total > 0, inside / body_total.clamp(min=1.0),
                        torch.zeros_like(body_total))
    rejected = ratio < STICKER_OVERLAP_MIN
    rms = torch.where(rejected, torch.full_like(rms, float("inf")), rms)

    return rms, src.round().to(torch.uint8)


def score_batch(
    masks: torch.Tensor,                       # (K, H, W) float in 0..1
    current_u8: torch.Tensor,                  # (H, W, 3) uint8
    target_u8: torch.Tensor,                   # (H, W, 3) uint8
    alpha_mask: torch.Tensor | None = None,    # (H, W) uint8 or None
    alpha: int = ALPHA_FIXED,
    alpha_levels: list[int] | None = None,
    edge_weight: torch.Tensor | None = None,   # (H, W) per-pixel loss weight or None
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Score K candidate masks. Returns (scores, colors, alphas).

    scores:  (K,) float  — RMS-if-committed (lower better). +inf for sticker-rejected shapes.
    colors:  (K, 3) uint8 — closed-form optimal RGB for the chosen alpha.
    alphas:  (K,) uint8  — the per-candidate alpha that minimized RMS.

    If `alpha_levels` is None, every candidate uses the single fixed `alpha` (the original
    behavior). If a list is given, each candidate is scored at every level and the alpha
    with the lowest RMS is kept per candidate — this lets a high-contrast dark feature pick
    a near-opaque alpha (crisp black lines) while a smooth gradient picks a low alpha.
    The optimal RGB is recomputed per alpha (it depends on alpha), so color and alpha stay
    consistent.
    """
    levels = alpha_levels if alpha_levels else [alpha]
    best_rms = best_col = best_a = None
    for a in levels:
        rms, col = _score_core(masks, current_u8, target_u8, alpha_mask, a, edge_weight)
        if best_rms is None:
            best_rms, best_col = rms, col
            best_a = torch.full_like(rms, float(a))
        else:
            better = rms < best_rms
            best_rms = torch.where(better, rms, best_rms)
            best_col = torch.where(better.unsqueeze(-1), col, best_col)
            best_a = torch.where(better, torch.full_like(rms, float(a)), best_a)
    return best_rms, best_col, best_a.round().to(torch.uint8)

# ----- shapes_gpu.py -----
"""Batched GPU shape kinds: hard + differentiable-soft rasterizers, random init, gradient
clamp, and JSON serialization for each supported shape type.

All rasterizers take a (K, P) param tensor (P depends on type) already on the device and
return a (K, H, W) mask in 0..1. The HARD rasterizer is used at commit time (crisp edges);
the SOFT rasterizer is differentiable w.r.t. params and used inside gradient refinement.

JSON output keys match the CPU shapes in forza_abyss_painter/shapegen/shapes/{ellipse,rectangle,triangle}.py
so the Windows FH6 injector loads them unchanged.
"""





# ----------------------------------------------------------------------------------------
# rotated_ellipse — params (x, y, rx, ry, angle)
# ----------------------------------------------------------------------------------------

def _ellipse_init(K, w, h, gen):
    out = torch.empty((K, 5), dtype=DTYPE)
    out[:, 0] = torch.rand(K, generator=gen) * w
    out[:, 1] = torch.rand(K, generator=gen) * h
    out[:, 2] = 1.0 + torch.rand(K, generator=gen) * (w / 8)
    out[:, 3] = 1.0 + torch.rand(K, generator=gen) * (h / 8)
    out[:, 4] = torch.rand(K, generator=gen) * 180.0
    return out


def _ellipse_soft(params, h, w):
    M = params.shape[0]
    dev = params.device
    x = params[:, 0].view(M, 1, 1); y = params[:, 1].view(M, 1, 1)
    rx = params[:, 2].clamp(min=1e-3).view(M, 1, 1); ry = params[:, 3].clamp(min=1e-3).view(M, 1, 1)
    ang = torch.deg2rad(params[:, 4]).view(M, 1, 1)
    ca, sa = torch.cos(ang), torch.sin(ang)
    ys = torch.arange(h, dtype=DTYPE, device=dev).view(1, h, 1)
    xs = torch.arange(w, dtype=DTYPE, device=dev).view(1, 1, w)
    dx = xs - x; dy = ys - y
    xr = ca * dx + sa * dy; yr = -sa * dx + ca * dy
    d2 = (xr / rx) ** 2 + (yr / ry) ** 2
    ramp = (2.0 / torch.minimum(rx, ry)).clamp(min=1e-3, max=0.5)
    return ((1.0 - d2 + ramp / 2.0) / ramp).clamp(0.0, 1.0)


def _ellipse_clamp_(p, w, h):
    p[:, 0].clamp_(0.0, w - 1); p[:, 1].clamp_(0.0, h - 1)
    p[:, 2].clamp_(1.0, float(w)); p[:, 3].clamp_(1.0, float(h))
    p[:, 4].remainder_(180.0)


def _ellipse_json(row, color):
    return {
        "type": "rotated_ellipse",
        "x": round(float(row[0]), 3), "y": round(float(row[1]), 3),
        "rx": round(float(row[2]), 3), "ry": round(float(row[3]), 3),
        "angle": round(float(row[4]), 3), "color": color,
    }


# ----------------------------------------------------------------------------------------
# rotated_rectangle — params (x, y, hw, hh, angle)
# ----------------------------------------------------------------------------------------

def _rect_init(K, w, h, gen):
    out = torch.empty((K, 5), dtype=DTYPE)
    out[:, 0] = torch.rand(K, generator=gen) * w
    out[:, 1] = torch.rand(K, generator=gen) * h
    out[:, 2] = 1.0 + torch.rand(K, generator=gen) * (w / 8)
    out[:, 3] = 1.0 + torch.rand(K, generator=gen) * (h / 8)
    out[:, 4] = torch.rand(K, generator=gen) * 180.0
    return out


def _rect_dist(params, h, w):
    """Chebyshev box distance d = max(|xr|/hw, |yr|/hh); inside iff d <= 1. (K, H, W)."""
    M = params.shape[0]
    dev = params.device
    x = params[:, 0].view(M, 1, 1); y = params[:, 1].view(M, 1, 1)
    hw = params[:, 2].clamp(min=1e-3).view(M, 1, 1); hh = params[:, 3].clamp(min=1e-3).view(M, 1, 1)
    ang = torch.deg2rad(params[:, 4]).view(M, 1, 1)
    ca, sa = torch.cos(ang), torch.sin(ang)
    ys = torch.arange(h, dtype=DTYPE, device=dev).view(1, h, 1)
    xs = torch.arange(w, dtype=DTYPE, device=dev).view(1, 1, w)
    dx = xs - x; dy = ys - y
    xr = ca * dx + sa * dy; yr = -sa * dx + ca * dy
    d = torch.maximum(xr.abs() / hw, yr.abs() / hh)
    ramp = (1.0 / torch.minimum(hw, hh)).clamp(min=1e-3, max=0.5)
    return d, ramp


def _rect_hard(params, h, w):
    d, _ = _rect_dist(params, h, w)
    return (d <= 1.0).to(DTYPE)


def _rect_soft(params, h, w):
    d, ramp = _rect_dist(params, h, w)
    # Linear edge centered at d=1, ~1 normalized-ramp wide.
    return ((1.0 - d) / ramp + 0.5).clamp(0.0, 1.0)


def _rect_json(row, color):
    return {
        "type": "rotated_rectangle",
        "x": round(float(row[0]), 3), "y": round(float(row[1]), 3),
        "hw": round(float(row[2]), 3), "hh": round(float(row[3]), 3),
        "angle": round(float(row[4]), 3), "color": color,
    }


# ----------------------------------------------------------------------------------------
# triangle — params (x1, y1, x2, y2, x3, y3)
# ----------------------------------------------------------------------------------------

def _tri_init(K, w, h, gen):
    spread = max(4.0, min(w, h) / 8.0)
    cx = (torch.rand(K, generator=gen) * w).view(K, 1)
    cy = (torch.rand(K, generator=gen) * h).view(K, 1)
    out = torch.empty((K, 6), dtype=DTYPE)
    for v in range(3):
        out[:, 2 * v] = (cx[:, 0] + torch.randn(K, generator=gen) * spread)
        out[:, 2 * v + 1] = (cy[:, 0] + torch.randn(K, generator=gen) * spread)
    out[:, 0::2].clamp_(0.0, w - 1)
    out[:, 1::2].clamp_(0.0, h - 1)
    return out


def _tri_signed(params, h, w):
    """Per-pixel inside-amount: min over the 3 edges of the winding-oriented perpendicular
    distance (pixels). >=0 inside. Differentiable w.r.t. vertices. Returns (K, H, W)."""
    M = params.shape[0]
    dev = params.device
    ax = params[:, 0].view(M, 1, 1); ay = params[:, 1].view(M, 1, 1)
    bx = params[:, 2].view(M, 1, 1); by = params[:, 3].view(M, 1, 1)
    cx = params[:, 4].view(M, 1, 1); cy = params[:, 5].view(M, 1, 1)
    ys = torch.arange(h, dtype=DTYPE, device=dev).view(1, h, 1)
    xs = torch.arange(w, dtype=DTYPE, device=dev).view(1, 1, w)

    def edge(p0x, p0y, p1x, p1y):
        # signed area*2 of (p0, p1, pixel); also = perpendicular dist * edge length
        e = (p1x - p0x) * (ys - p0y) - (p1y - p0y) * (xs - p0x)
        length = torch.sqrt((p1x - p0x) ** 2 + (p1y - p0y) ** 2).clamp(min=1e-3)
        return e / length   # (M, H, W) perpendicular distance (signed)

    d_ab = edge(ax, ay, bx, by)
    d_bc = edge(bx, by, cx, cy)
    d_ca = edge(cx, cy, ax, ay)
    # winding sign from the triangle's own area (orient so inside is positive)
    area2 = (bx - ax) * (cy - ay) - (by - ay) * (cx - ax)   # (M,1,1)
    s = torch.where(area2 >= 0, torch.ones_like(area2), -torch.ones_like(area2))
    inside = torch.minimum(torch.minimum(s * d_ab, s * d_bc), s * d_ca)
    return inside


def _tri_hard(params, h, w):
    return (_tri_signed(params, h, w) >= 0.0).to(DTYPE)


def _tri_soft(params, h, w):
    inside = _tri_signed(params, h, w)
    return (inside + 0.5).clamp(0.0, 1.0)   # ~1px AA centered at the boundary


def _tri_clamp_(p, w, h):
    p[:, 0::2].clamp_(0.0, w - 1)   # x coords
    p[:, 1::2].clamp_(0.0, h - 1)   # y coords


def _tri_json(row, color):
    return {
        "type": "triangle",
        "x1": round(float(row[0]), 3), "y1": round(float(row[1]), 3),
        "x2": round(float(row[2]), 3), "y2": round(float(row[3]), 3),
        "x3": round(float(row[4]), 3), "y3": round(float(row[5]), 3),
        "color": color,
    }


# ----------------------------------------------------------------------------------------
# Registry
# ----------------------------------------------------------------------------------------

@dataclass(frozen=True)
class ShapeKind:
    name: str
    param_count: int
    init: Callable          # (K, w, h, gen) -> (K, P) cpu tensor
    rasterize_hard: Callable  # (params, h, w) -> (K, H, W)
    rasterize_soft: Callable  # (params, h, w) -> (K, H, W) differentiable
    clamp_: Callable        # (params, w, h) -> in-place
    to_json: Callable       # (row_list, color_list) -> dict


KINDS: dict[str, ShapeKind] = {
    "rotated_ellipse": ShapeKind(
        "rotated_ellipse", 5, _ellipse_init,
        rasterize_rotated_ellipses, _ellipse_soft, _ellipse_clamp_, _ellipse_json),
    "rotated_rectangle": ShapeKind(
        "rotated_rectangle", 5, _rect_init,
        _rect_hard, _rect_soft, _ellipse_clamp_, _rect_json),
    "triangle": ShapeKind(
        "triangle", 6, _tri_init,
        _tri_hard, _tri_soft, _tri_clamp_, _tri_json),
}

# ----- bbox_score.py -----
"""Bounding-box-local scoring for the random-search phase (ellipse-only).

The full-canvas scorer materializes (K, H, W, 3) intermediates, capping K at ~1-2k. But a
shape only touches its own bounding box; pixels outside are unchanged. So we can score each
candidate over a small per-candidate crop and compare candidates by the *incremental* SSE
delta they produce inside their box:

    delta_k = SSE_after_in_box(k) - SSE_before_in_box(k)

argmin over k of delta_k is the globally-best shape (the global SSE-before is identical for
all candidates, and each shape changes only its own box). This is geometrize's incremental
energy, batched on GPU. Crops are ~10x+ smaller than the full canvas, so the random phase
can run ~10x+ more candidates for the same VRAM — the geometrize "raise random samples"
lever, made affordable.

v1 uses a single crop size for the batch (= the batch's max ellipse radius), so the box
always contains every candidate's mask. Size-binning (smaller crops for small shapes → even
more samples) is a future refinement. Ellipse-only; gradient refinement stays full-canvas.
"""




def crop_score_ellipse_batch(
    params: torch.Tensor,            # (K, 5) ellipse params on device
    canvas_u8: torch.Tensor,         # (H, W, 3) uint8
    target_u8: torch.Tensor,         # (H, W, 3) uint8
    alpha_t: torch.Tensor | None = None,    # (H, W) uint8 sticker mask or None
    edge_weight: torch.Tensor | None = None,  # (H, W) per-pixel loss weight or None
    alpha_levels: list[int] | None = None,
    max_crop_radius: int = 256,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Returns (delta_scores (K,), colors_u8 (K,3), alphas_u8 (K,)).

    delta_scores: in-bbox SSE change if the candidate were committed (lower = better;
                  typically negative). +inf for sticker-rejected candidates. argmin over K
                  picks the globally best candidate.
    """
    K = params.shape[0]
    device = params.device
    H, W = canvas_u8.shape[:2]

    # Crop half-size E = batch's max radius (so every candidate's mask fits the box),
    # capped at max_crop_radius. B = 2E+1.
    rmax = float(params[:, 2:4].max().clamp(min=1.0).ceil().item())
    E = min(int(max_crop_radius), int(rmax) + 1)
    B = 2 * E + 1

    cur = canvas_u8.to(DTYPE)
    tgt = target_u8.to(DTYPE)

    cx_i = params[:, 0].round().long()
    cy_i = params[:, 1].round().long()
    ox = cx_i - E                                   # (K,) crop origin (top-left)
    oy = cy_i - E
    lin = torch.arange(B, device=device)
    gx = ox.view(K, 1) + lin.view(1, B)             # (K, B) global col idx
    gy = oy.view(K, 1) + lin.view(1, B)             # (K, B) global row idx
    vx = (gx >= 0) & (gx < W)                       # (K, B) col validity
    vy = (gy >= 0) & (gy < H)
    gxc = gx.clamp(0, W - 1)
    gyc = gy.clamp(0, H - 1)
    valid = (vy.unsqueeze(2) & vx.unsqueeze(1)).to(DTYPE)   # (K, B, B)

    row_idx = gyc.unsqueeze(2)                       # (K, B, 1)
    col_idx = gxc.unsqueeze(1)                       # (K, 1, B)
    cur_c = cur[row_idx, col_idx]                    # (K, B, B, 3)
    tgt_c = tgt[row_idx, col_idx]

    # Hard ellipse mask in crop coords (global pixel coords = origin + local index).
    cxp = params[:, 0].view(K, 1, 1)
    cyp = params[:, 1].view(K, 1, 1)
    rx = params[:, 2].clamp(min=1e-3).view(K, 1, 1)
    ry = params[:, 3].clamp(min=1e-3).view(K, 1, 1)
    ang = torch.deg2rad(params[:, 4]).view(K, 1, 1)
    ca, sa = torch.cos(ang), torch.sin(ang)
    lin_f = lin.to(DTYPE)
    gxf = ox.to(DTYPE).view(K, 1, 1) + lin_f.view(1, 1, B)   # (K, 1, B)
    gyf = oy.to(DTYPE).view(K, 1, 1) + lin_f.view(1, B, 1)   # (K, B, 1)
    dxc = gxf - cxp
    dyc = gyf - cyp
    xr = ca * dxc + sa * dyc
    yr = -sa * dxc + ca * dyc
    d2 = (xr / rx) ** 2 + (yr / ry) ** 2             # (K, B, B)
    mask = (d2 <= 1.0).to(DTYPE) * valid             # zero outside canvas

    # Per-pixel loss weight: opaque gate (sticker) × edge emphasis × validity, folded WITH
    # the shape mask into a single (K,B,B,1) combined weight `mw3` — this is multiplied into
    # every SSE term, so folding it once avoids holding mask3 + pw3 separately (each
    # (K,B,B,1) tensor is sizeable at high K). Frees intermediates aggressively because at
    # high K every (K,B,B,3) tensor is large; minimizing how many are live at once is what
    # keeps the bbox path within VRAM.
    if alpha_t is not None:
        alpha_c = alpha_t.to(DTYPE)[row_idx, col_idx]    # (K, B, B) 0..255
        pw = (alpha_c > 0).to(DTYPE)
    else:
        alpha_c = None
        pw = torch.ones((K, B, B), dtype=DTYPE, device=device)
    if edge_weight is not None:
        pw = pw * edge_weight[row_idx, col_idx]
    pw = pw * valid
    mw3 = (mask * pw).unsqueeze(-1)                       # (K,B,B,1) combined mask×weight
    del pw

    diff = cur_c - tgt_c
    sse_old = (mw3 * diff.square()).sum(dim=(1, 2, 3))    # (K,)
    del diff

    # Sticker overlap rejection (body must sit inside the opaque region).
    if alpha_t is not None:
        body = mask >= _BODY_THRESHOLD
        body_total = body.sum(dim=(1, 2)).to(DTYPE)
        opaque_body = (alpha_c >= 128) & (valid > 0)
        inside = (body & opaque_body).sum(dim=(1, 2)).to(DTYPE)
        ratio = torch.where(body_total > 0, inside / body_total.clamp(min=1.0),
                            torch.zeros_like(body_total))
        rejected = ratio < STICKER_OVERLAP_MIN
    else:
        rejected = torch.zeros(K, dtype=torch.bool, device=device)
    del valid

    best_delta = best_col = best_a = None
    for alpha in (alpha_levels or [ALPHA_FIXED]):
        a = float(alpha) / 255.0
        eff = mask * (alpha_c / 255.0) if alpha_t is not None else mask   # color weight
        eff3 = eff.unsqueeze(-1)
        wsum = eff.sum(dim=(1, 2)).clamp(min=1e-3)
        sum_mt = (eff3 * tgt_c).sum(dim=(1, 2))
        sum_mc = (eff3 * cur_c).sum(dim=(1, 2))
        del eff, eff3
        src = ((sum_mt - (1.0 - a) * sum_mc) / (a * wsum.unsqueeze(-1))).clamp(0.0, 255.0)
        diff_new = a * src.view(K, 1, 1, 3) + (1.0 - a) * cur_c - tgt_c
        sse_new = (mw3 * diff_new.square()).sum(dim=(1, 2, 3))
        del diff_new
        delta = sse_new - sse_old
        if best_delta is None:
            best_delta, best_col = delta, src
            best_a = torch.full((K,), float(alpha), dtype=DTYPE, device=device)
        else:
            better = delta < best_delta
            best_delta = torch.where(better, delta, best_delta)
            best_col = torch.where(better.unsqueeze(-1), src, best_col)
            best_a = torch.where(better, torch.full_like(best_a, float(alpha)), best_a)

    best_delta = torch.where(rejected, torch.full_like(best_delta, float("inf")), best_delta)
    return best_delta, best_col.round().to(torch.uint8), best_a.round().to(torch.uint8)

# ----- joint_polish.py -----
"""Joint-polish pass: diffvg-style global refinement of all shapes at once.

Greedy generation (geometrize and ours alike) freezes each shape the moment it's placed, so
early shapes are wrong in hindsight and later shapes waste budget compensating. This pass
attacks that ceiling: after greedy placement, it makes the *whole* composite differentiable
and gradient-optimizes ALL shapes' geometry + color + alpha simultaneously against the full
target. One optimization run fixes errors that greedy would need many more shapes to paper
over — more quality per shape, without geometrize's brute-force sample counts.

Memory: N sequential soft-composites would store N intermediate canvases for backward. We
chunk the composite and gradient-checkpoint each chunk (recompute its forward during
backward), so peak memory is O(num_chunks + chunk_size) canvases, not O(N).

Ellipse-only (matches the shippable shape set).
"""

import torch.utils.checkpoint as cp


_ALPHA_MIN = 16.0   # don't let opacities collapse to invisible during optimization


def _composite_chunk(canvas, geom_c, rgb_c, alpha_c, h, w):
    """Soft-composite a chunk of shapes (in order) onto `canvas`. Differentiable.
    geom_c (C,5), rgb_c (C,3) in 0..255, alpha_c (C,) in 0..255.

    No silhouette clipping: shapes paint their full mask. The in-game and CPU renderers have
    no source alpha mask to clip against, so we don't either — the optimizer trains against
    what the game will actually display. Spill penalty is encoded into the target instead
    (engine pre-fills out-of-silhouette target pixels with the canvas substrate color, so
    paint that lands there shows up as positive loss against an otherwise-zero baseline).
    """
    masks = _ellipse_soft(geom_c, h, w)          # (C, H, W), soft/differentiable
    C = geom_c.shape[0]
    for i in range(C):
        m = masks[i]
        a = (alpha_c[i] / 255.0).clamp(0.0, 1.0)
        src = rgb_c[i].clamp(0.0, 255.0).view(1, 1, 3)
        mm = m.unsqueeze(-1)
        canvas = mm * (a * src + (1.0 - a) * canvas) + (1.0 - mm) * canvas
    return canvas


def _purity_loss(geom, tgt, h, w, chunk):
    """Per-shape MSE-equivalent spillover loss, summed over shapes, normalized by canvas pixels.

    For each shape i with soft mask m_i (H, W) and target t (H, W, 3):
        mass_i        = sum(m_i)
        mean_i        = sum(m_i * t) / mass_i                        (3,)
        spillover_i   = sum( m_i * (t - mean_i).pow(2) ).sum()       scalar (RGB-summed)
        # spillover_i ≡ the MSE the canvas would carry over shape i's mask if i painted
        # with its optimal mean color — equal to mass_i × per-pixel-variance.

    Returns: sum_i(spillover_i) / (H * W * 3), in the same MSE-equivalent units as the
    main loss above. This is the load-bearing change vs the original PR #19 formulation
    (which returned per-shape AVERAGE variance — mass-independent — so a tiny lazy shape
    and a huge lazy shape paid the same penalty while their MSE gradients differed by
    orders of magnitude). The mass-weighted form makes big lazy shapes pay proportionally
    more, which is exactly what we want: the optimizer's incentive to grow a shape (more
    pixels of MSE benefit) is now balanced by a proportionally-growing penalty.

    Intuition for tuning purity_penalty:
      = 1.0   → lazy paint penalty equals its MSE saving; lazy and no-paint break even
      < 1.0   → painting still preferred to no-paint, but homogeneity strongly encouraged
      > 1.0   → no-paint strictly preferred to lazy paint over the same multi-color region

    Chunked so we never materialize all N masks at once at production shape counts.
    Fully differentiable wrt geom.
    """
    N = geom.shape[0]
    if N == 0:
        return torch.zeros((), dtype=DTYPE, device=geom.device)
    spillover_sum = torch.zeros((), dtype=DTYPE, device=geom.device)
    for lo in range(0, N, chunk):
        hi = min(lo + chunk, N)
        masks = _ellipse_soft(geom[lo:hi], h, w)              # (C, H, W)
        mass = masks.sum(dim=(1, 2)).clamp(min=1e-6)          # (C,)
        weighted = (masks.unsqueeze(-1) * tgt.unsqueeze(0)).sum(dim=(1, 2))   # (C, 3)
        mean_color = weighted / mass.unsqueeze(-1)            # (C, 3)
        diff = tgt.unsqueeze(0) - mean_color.view(-1, 1, 1, 3)               # (C, H, W, 3)
        # Mass-weighted (NOT mass-normalized) sum of squared deviations per shape.
        # This is `mass * variance_per_pixel` for each shape, RGB-summed.
        spillover_per_shape = (masks.unsqueeze(-1) * diff.pow(2)).sum(dim=(1, 2, 3))  # (C,)
        spillover_sum = spillover_sum + spillover_per_shape.sum()
    return spillover_sum / float(h * w * 3)


def _forward_composite(canvas0, geom, rgb, alpha, h, w, chunk, use_ckpt):
    canvas = canvas0
    N = geom.shape[0]
    for lo in range(0, N, chunk):
        hi = min(lo + chunk, N)
        g, c, a = geom[lo:hi], rgb[lo:hi], alpha[lo:hi]
        if use_ckpt and torch.is_grad_enabled():
            canvas = cp.checkpoint(_composite_chunk, canvas, g, c, a,
                                   h, w, use_reentrant=False)
        else:
            canvas = _composite_chunk(canvas, g, c, a, h, w)
    return canvas


def _hard_render(canvas0, geom, rgb, alpha, h, w):
    """Final crisp render with hard ellipse edges. UNCLIPPED — matches what the in-game and
    CPU exe renderers produce for the emitted JSON. (Was previously clipping by alpha_mask_f
    in sticker mode, which produced a clean preview that diverged from the actual game render
    and hid out-of-silhouette spillover. See _composite_chunk docstring.)"""
    canvas = canvas0.clone()
    N = geom.shape[0]
    for i in range(N):
        mask = rasterize_rotated_ellipses(geom[i:i + 1], h, w)[0]
        a = float(alpha[i].clamp(0.0, 255.0).item()) / 255.0
        src = rgb[i].clamp(0.0, 255.0).view(1, 1, 3)
        m = mask.unsqueeze(-1)
        canvas = m * (a * src + (1.0 - a) * canvas) + (1.0 - m) * canvas
    return canvas


def _resolve_rgb_closed_form(geom, target, alpha_mask_f, h, w):
    """Closed-form post-polish RGB resolution for lock_alpha=True.

    For each shape i (in commit order), its actually-visible pixels are:
        mask_i AND (opaque substrate, if sticker) AND (no later shape covers it)

    Since alpha=255 fully overwrites within mask_i, shape_i's contribution to the rendered
    image is exactly the pixels it owns at the top of the z-stack. The closed-form optimal
    color (minimizing per-pixel SSE against the target there) is the simple mean of the
    target's RGB over those visible pixels.

    Walks shapes in REVERSE order so we can accumulate the "claimed" union from front to
    back. Fully-occluded shapes (mask entirely covered by later layers) fall back to the
    mean over their full mask — preserves whatever color they had if they ever get
    un-occluded by future edits.
    """
    n = geom.shape[0]
    device = geom.device
    target_f = target.to(DTYPE)   # (H, W, 3)

    new_rgb = torch.zeros((n, 3), dtype=DTYPE, device=device)
    claimed = torch.zeros((h, w), dtype=torch.bool, device=device)
    opaque = (alpha_mask_f > 0) if alpha_mask_f is not None else None

    fallback_color = torch.tensor([128.0, 128.0, 128.0], dtype=DTYPE, device=device)
    for i in reversed(range(n)):
        mask = rasterize_rotated_ellipses(geom[i:i + 1], h, w)[0]   # (H, W) {0, 1}
        m_bool = mask > 0
        visible = m_bool & ~claimed
        if opaque is not None:
            visible = visible & opaque
        if visible.any():
            new_rgb[i] = target_f[visible].mean(dim=0)
        elif m_bool.any():
            # Fully occluded — keep a sane in-distribution color via the all-mask mean
            # (still beats Adam saturation extremes).
            new_rgb[i] = target_f[m_bool].mean(dim=0)
        else:
            new_rgb[i] = fallback_color
        claimed |= m_bool
    return new_rgb


def joint_polish(shapes_json, target, alpha_t, alpha_mask_f, edge_weight, canvas_init,
                 h, w, steps, lr=1.5, chunk=200, progress=False, lock_alpha=True,
                 purity_penalty=0.0, freeze_geometry=True):
    """Jointly refine all shapes. Returns (refined_shapes_json, final_canvas_u8_numpy).

    target/canvas_init: (H,W,3) uint8 device tensors (target already posterized + sticker-
    zeroed by the caller; canvas_init is the same init the greedy run used).
    alpha_t (H,W uint8) / alpha_mask_f (H,W float) for sticker; edge_weight (H,W) or None.

    purity_penalty (float, default 0.0): when > 0 AND freeze_geometry is False, adds a
    per-shape MSE-equivalent 'spillover' loss (mass × per-pixel variance, summed over
    shapes, normalized by canvas pixel count — same units as the main MSE). At
    purity_penalty=1.0, lazy paint cancels its MSE saving (break-even with omission).
    Empirically the penalty creates a degenerate exploit (Adam collapses shapes to rx=1
    to escape the penalty), so freeze_geometry=True is the recommended polish mode.

    freeze_geometry (bool, default False): when True, Adam optimizes (rgb, alpha) ONLY —
    geometry (x, y, rx, ry, angle) stays bit-identical to the input shapes_json. This is
    the recommended polish mode: it keeps the color refinement + snap-back win without
    giving Adam the geometry handles it consistently mis-uses (inflate, collapse, drift).
    purity_penalty is a no-op when freeze_geometry=True (it's a geometry-affecting term).
    """
    device = target.device
    if not shapes_json:
        return shapes_json, _hard_render(canvas_init.to(DTYPE), torch.zeros((0, 5), device=device),
                                         torch.zeros((0, 3), device=device),
                                         torch.zeros((0,), device=device),
                                         h, w).clamp(0, 255).round().to(torch.uint8).cpu().numpy()

    geom = torch.tensor([[s["x"], s["y"], s["rx"], s["ry"], s["angle"]] for s in shapes_json],
                        dtype=DTYPE, device=device)
    rgb = torch.tensor([s["color"][:3] for s in shapes_json], dtype=DTYPE, device=device)
    alpha = torch.tensor([s["color"][3] for s in shapes_json], dtype=DTYPE, device=device)

    if not freeze_geometry:
        geom.requires_grad_(True)
    rgb.requires_grad_(True)
    if lock_alpha:
        alpha.fill_(255.0)   # force the locked value even if input had alpha < 255
    else:
        alpha.requires_grad_(True)
    canvas0 = canvas_init.to(DTYPE)
    tgt = target.to(DTYPE)

    # Per-pixel loss weight: edge emphasis only. NO opaque (alpha_t) gate — that previously
    # zeroed out-of-silhouette pixels in the loss, which prevented the optimizer from feeling
    # spillover past the silhouette. The caller pre-fills the target with the canvas substrate
    # color outside the silhouette (see engine.py:341), so out-of-mask paint is now naturally
    # penalized by plain MSE without any gating. lw stays None unless edge_weight is provided.
    lw = edge_weight
    if lw is not None:
        lw3 = lw.unsqueeze(-1)
        denom = (lw.sum() * 3.0).clamp(min=1.0)

    opt_groups = [{"params": [rgb], "lr": lr * 4.0}]   # colors always optimized
    if not freeze_geometry:
        opt_groups.append({"params": [geom], "lr": lr})
    if not lock_alpha:
        opt_groups.append({"params": [alpha], "lr": lr * 4.0})
    opt = torch.optim.Adam(opt_groups)

    init_loss = None
    for step in range(steps):
        opt.zero_grad()
        canvas = _forward_composite(canvas0, geom, rgb, alpha, h, w,
                                    chunk, use_ckpt=True)
        diff = canvas - tgt
        if lw is not None:
            loss = (diff * diff * lw3).sum() / denom
        else:
            loss = (diff * diff).mean()
        # purity_penalty is a GEOMETRY-affecting penalty (the only way to reduce it is to
        # change shape masks, i.e. geometry). When freeze_geometry is on, geom has no grad,
        # so the penalty would only inflate the loss without producing any update — skip it.
        if purity_penalty > 0.0 and not freeze_geometry:
            loss = loss + purity_penalty * _purity_loss(geom, tgt, h, w, chunk)
        loss.backward()
        opt.step()
        with torch.no_grad():
            if not freeze_geometry:
                geom[:, 0].clamp_(0.0, w - 1)
                geom[:, 1].clamp_(0.0, h - 1)
                geom[:, 2].clamp_(1.0, float(w))
                geom[:, 3].clamp_(1.0, float(h))
            rgb.clamp_(0.0, 255.0)
            if not lock_alpha:
                alpha.clamp_(_ALPHA_MIN, 255.0)
        if init_loss is None:
            init_loss = float(loss.detach().cpu())
        if progress and (step + 1) % max(1, steps // 5) == 0:
            print(f"  joint-polish step {step+1}/{steps}  loss {float(loss.detach().cpu()):.2f}")

    geom_f = geom.detach(); rgb_f = rgb.detach(); alpha_f = alpha.detach()
    with torch.no_grad():
        # Color snap-back. Adam runs RGB at lr*4 across `steps` updates with no chromatic
        # regularization — momentum routinely pushes channels to saturation extremes that the
        # underlying target image doesn't contain. (Observed: ~125/3000 shapes hitting
        # min<15 AND max>240 in 150-step polish, with ~45 turning fully green/yellow when
        # the optimizer overshot from sparkle-yellow target regions.) For each polished
        # shape, recompute the closed-form-optimal opaque color as the MEAN of the target
        # image over the shape's actually-visible mask region (mask AND opaque substrate
        # AND not yet claimed by a later layer in z-order). This is the exact value Adam was
        # approximating and it stays in the target's color distribution by construction.
        if lock_alpha:
            rgb_f = _resolve_rgb_closed_form(geom_f, target.detach(), alpha_mask_f, h, w)
        final_canvas = _hard_render(canvas0, geom_f, rgb_f, alpha_f, h, w)
        final_canvas = final_canvas.clamp(0, 255).round().to(torch.uint8).cpu().numpy()

    g = geom_f.cpu().tolist(); c = rgb_f.round().cpu().tolist(); al = alpha_f.round().cpu().tolist()
    refined = [{
        "type": "rotated_ellipse",
        "x": round(float(g[i][0]), 3), "y": round(float(g[i][1]), 3),
        "rx": round(float(g[i][2]), 3), "ry": round(float(g[i][3]), 3),
        "angle": round(float(g[i][4]) % 180.0, 3),
        "color": [int(c[i][0]), int(c[i][1]), int(c[i][2]),
                  255 if lock_alpha else int(al[i])],
    } for i in range(len(shapes_json))]
    return refined, final_canvas

# ----- refill.py -----
"""Post-greedy shape-budget reclamation.

THE PROBLEM: greedy commits N shapes one at a time, each one optimal at its commit
moment. Later commits can fully occlude earlier ones (especially in regions where the
target has fine detail that takes multiple shapes to capture). At production budgets
(1000-3000 shapes) we measure ~5% of commits ending up fully-occluded — pure dead
weight in the final JSON. The user asked for N shapes; the engine delivers N commits
but only ~0.95 N actually contribute pixels to the final canvas.

THE FIX (option B from the design discussion):
After greedy + polish + snap-back completes, run a focused cleanup-and-refill pass:

  1. Detect dead shapes (visible-pixel count below threshold under z-order)
  2. Drop them from the shape list
  3. Re-render the canvas from the surviving live shapes
  4. Mini-greedy: commit N_dead NEW shapes targeting the current residual (the parts
     of the canvas that diverge from the target). Same bbox-local + gradient-refine
     logic as the main greedy loop, just bounded to N_dead iterations and starting
     from the post-cleanup canvas state.
  5. Closed-form RGB snap-back over the full final shape set (existing + new), so
     each shape's color is the mean of target over its visible region in the new
     occlusion structure.
  6. Re-render final canvas via _hard_render.

User asks for 3000; gets ~3000 LIVE shapes that all contribute pixels to the final
render. The 5% waste that was happening invisibly in the JSON gets re-allocated to
useful work in high-residual regions.

This is essentially the polish_existing pipeline from earlier sessions, but cleaner
and integrated as a default-on stage in run_gpu rather than a separate notebook flow.
"""




def _identify_dead_shape_indices(shapes_json: list[dict], h: int, w: int,
                                  device, min_visible_px: int) -> set[int]:
    """Walk shapes in REVERSE z-order (topmost first). For each, the visible pixel set
    is its mask AND NOT-yet-claimed. Accumulate `claimed` from top down. Return indices
    whose visible-pixel count is below threshold (= effectively invisible in the final
    composite under alpha=255 z-order)."""
    n = len(shapes_json)
    if n == 0:
        return set()
    geom = torch.tensor(
        [[s["x"], s["y"], s["rx"], s["ry"], s["angle"]] for s in shapes_json],
        dtype=DTYPE, device=device,
    )
    claimed = torch.zeros((h, w), dtype=torch.bool, device=device)
    dead: set[int] = set()
    for i in reversed(range(n)):
        mask = rasterize_rotated_ellipses(geom[i:i + 1], h, w)[0] > 0
        visible = mask & ~claimed
        if int(visible.sum().cpu()) < min_visible_px:
            dead.add(i)
        claimed |= mask
    return dead


def _render_live_canvas(shapes_json: list[dict], canvas_init: torch.Tensor,
                         h: int, w: int, device) -> torch.Tensor:
    """Hard-render the (live, in-order) shapes onto a fresh canvas. Returns DTYPE
    tensor of shape (H, W, 3) — same format the greedy loop maintains."""
    if not shapes_json:
        return canvas_init.to(DTYPE)
    geom = torch.tensor(
        [[s["x"], s["y"], s["rx"], s["ry"], s["angle"]] for s in shapes_json],
        dtype=DTYPE, device=device,
    )
    rgb = torch.tensor([s["color"][:3] for s in shapes_json], dtype=DTYPE, device=device)
    alpha = torch.tensor([s["color"][3] for s in shapes_json], dtype=DTYPE, device=device)
    return _hard_render(canvas_init.to(DTYPE), geom, rgb, alpha, h, w)


def _mini_greedy_refill(canvas: torch.Tensor, target: torch.Tensor,
                        alpha_t, alpha_mask_f, edge_weight,
                        n_to_commit: int, cfg, h: int, w: int,
                        generator) -> tuple[list[dict], torch.Tensor]:
    """Commit exactly n_to_commit new rotated_ellipse shapes targeting the current
    residual (canvas vs target). Self-contained mini-greedy using the same bbox-local
    + gradient-refine primitives as the main greedy loop in engine.py, but bounded to
    a fixed iteration count and starting from the post-cleanup canvas.

    NOTE: ellipse-only, gradient-refine-only path — matches the production preset
    config (bbox_local=True, refine_mode='gradient', shape_types=['rotated_ellipse']).
    If extending refill to multi-shape or hill-climb modes in the future, expand here.
    """
    # Lazy import to avoid circular dependency: engine.py imports from this module
    # (well, run_gpu calls clean_and_refill) and this needs _refine_gradient + _composite_one
    # from engine. Lazy import sidesteps that during module load.

    ekind = KINDS["rotated_ellipse"]
    new_shapes: list[dict] = []
    canvas_u8 = canvas.clamp(0, 255).round().to(torch.uint8)
    target_u8 = target.to(torch.uint8) if target.dtype != torch.uint8 else target
    device = canvas.device

    for _ in range(n_to_commit):
        # 1. Random search over K candidates targeting the residual canvas.
        params = ekind.init(cfg.random_samples, w, h, generator).to(device)
        scores, colors, alphas = crop_score_ellipse_batch(
            params, canvas_u8, target_u8,
            alpha_t=alpha_t, edge_weight=edge_weight,
            alpha_levels=cfg.alpha_levels, max_crop_radius=cfg.bbox_crop_max,
        )
        if not torch.isfinite(scores).any():
            break   # all candidates rejected (eg sticker overlap too low); nothing useful to add
        best_idx = int(scores.argmin().cpu().item())
        best_score = float(scores[best_idx].cpu().item())
        if best_score == float("inf"):
            break

        # 2. Gradient-refine top-M candidates.
        M = min(cfg.grad_starts, params.shape[0])
        top_idx = torch.topk(scores, M, largest=False).indices
        init = params[top_idx].clone()
        best_params, best_color, best_alpha, refined_score = _refine_gradient(
            ekind, init, canvas_u8, target_u8, alpha_t, alpha_mask_f, h, w,
            cfg.grad_steps, cfg.grad_lr, alpha_levels=cfg.alpha_levels,
            edge_weight=edge_weight,
        )
        if refined_score == float("inf"):
            break

        # 3. Commit.
        canvas_u8 = _composite_one(ekind, canvas_u8, best_params, best_color, h, w,
                                    alpha_mask_f, best_alpha)
        c = best_color.cpu().tolist()
        color = [int(c[0]), int(c[1]), int(c[2]), int(best_alpha)]
        new_shapes.append(ekind.to_json(best_params.cpu().tolist(), color))

    return new_shapes, canvas_u8.to(DTYPE)


def clean_and_refill(
    shapes_json: list[dict],
    target_rgb: torch.Tensor,           # (H, W, 3) uint8 or DTYPE tensor
    canvas_init: torch.Tensor,           # the canvas-init the engine started from
    alpha_t, alpha_mask_f, edge_weight,
    cfg, h: int, w: int, generator,
) -> tuple[list[dict], torch.Tensor]:
    """Drop dead shapes from `shapes_json`, refill the freed slots with new mini-greedy
    commits targeting current residual, snap-back colors. Returns (new_shapes_list,
    final_canvas_uint8_numpy).

    No-op if cfg.refill_dead_shapes is False or no dead shapes are detected — in that
    case the input shapes pass through and the canvas is just re-rendered for return.

    Caller is expected to have already run greedy + joint_polish + snap-back; this is
    the final stage before serialization."""
    if not shapes_json:
        # Empty input — return empty canvas.
        empty = canvas_init.to(DTYPE).clamp(0, 255).round().to(torch.uint8).cpu().numpy()
        return [], empty

    # Caller (engine.run_gpu) is expected to pre-gate on ellipse-only — the dead-shape
    # detection here uses the rotated_ellipse rasterizer and the mini-greedy commits
    # rotated_ellipses. Defensive assert for clarity if someone wires this in elsewhere.
    assert all(s["type"] == "rotated_ellipse" for s in shapes_json), (
        "clean_and_refill is ellipse-only; caller must gate on shape type"
    )

    device = canvas_init.device

    # 1. Detect dead shapes.
    dead_indices = _identify_dead_shape_indices(
        shapes_json, h, w, device, cfg.refill_min_visible_px,
    )
    if not dead_indices:
        # No waste — render whatever's in shapes_json and return.
        canvas = _render_live_canvas(shapes_json, canvas_init, h, w, device)
        return shapes_json, canvas.clamp(0, 255).round().to(torch.uint8).cpu().numpy()

    # 2. Drop dead, rebuild live canvas.
    live_shapes = [s for i, s in enumerate(shapes_json) if i not in dead_indices]
    n_dead = len(dead_indices)
    canvas_after_cleanup = _render_live_canvas(live_shapes, canvas_init, h, w, device)

    # 3. Mini-greedy: refill the freed slots.
    new_shapes, canvas_after_refill = _mini_greedy_refill(
        canvas_after_cleanup, target_rgb,
        alpha_t, alpha_mask_f, edge_weight,
        n_dead, cfg, h, w, generator,
    )

    # 4. Combine + closed-form RGB snap-back over the full set (so existing shapes get
    #    their colors re-tuned for the new occlusion structure after dead-drop + refill,
    #    and new shapes get their snap-back color too).
    full_shapes = live_shapes + new_shapes
    if cfg.lock_alpha and full_shapes:
        geom_full = torch.tensor(
            [[s["x"], s["y"], s["rx"], s["ry"], s["angle"]] for s in full_shapes],
            dtype=DTYPE, device=device,
        )
        target_f = target_rgb.to(DTYPE)
        rgb_new = _resolve_rgb_closed_form(geom_full, target_f, alpha_mask_f, h, w)
        # Write the snapped colors back into the shape dicts (preserve alpha=255).
        rgb_list = rgb_new.round().clamp(0, 255).cpu().tolist()
        for i, s in enumerate(full_shapes):
            s["color"] = [int(rgb_list[i][0]), int(rgb_list[i][1]),
                          int(rgb_list[i][2]), s["color"][3]]
        # Final render uses the snapped colors.
        alpha_full = torch.tensor(
            [s["color"][3] for s in full_shapes], dtype=DTYPE, device=device,
        )
        final_canvas = _hard_render(canvas_init.to(DTYPE), geom_full, rgb_new,
                                     alpha_full, h, w)
    else:
        # No snap-back when lock_alpha is off (which shouldn't happen given v0.1.7's gate,
        # but defensive). Use the mini-greedy's committed canvas as-is.
        final_canvas = canvas_after_refill

    final_u8 = final_canvas.clamp(0, 255).round().to(torch.uint8).cpu().numpy()
    return full_shapes, final_u8

# ----- engine.py -----
# Matches MAX_CONSECUTIVE_SKIPS in forza_abyss_painter/shapegen/engine.py:134
_MAX_CONSECUTIVE_SKIPS = 80

# Mutation sigmas (image-pixel units / degrees). From RotatedEllipse.mutate
# in forza_abyss_painter/shapegen/shapes/ellipse.py:112 — coarse single-axis + fine xy/angle modes.
_MUT_SIGMA_XY_COARSE = 16.0
_MUT_SIGMA_R_COARSE = 16.0
_MUT_SIGMA_ANGLE = 25.0
_MUT_SIGMA_XY_FINE = 8.0
_MUT_SIGMA_ANGLE_FINE = 15.0

# Soft penalty weight for paint bleeding outside the opaque region during gradient
# refinement. Bleed term is in pixel units; the MSE term is a per-pixel mean, so this
# needs to be O(1) to actually bite. Larger = shapes pushed harder to stay inside.
STICKER_GRAD_PENALTY = 1.0


@dataclass
class GPUConfig:
    """One-shot GPU engine config.

    refine_mode:
      "gradient"  — differentiable rasterizer + Adam (sharper, more sample-efficient)
      "hillclimb" — parallel mutation rounds (older path, kept for A/B comparison)

    Memory: score_batch materializes (K, H, W, 3) intermediates; peak ~ K*H*W*12 bytes,
    where K is random_samples (the seed batch). Gradient refinement only touches
    grad_starts candidates, so it's cheap on memory.
    """
    num_shapes: int = 500
    random_samples: int = 256
    seed: int = 0   # 0 → time-based
    refine_mode: str = "gradient"
    # Candidate shape types (gradient mode tries all and picks the best per shape).
    # hillclimb mode is rotated_ellipse-only (no per-kind mutation defined for others).
    shape_types: list[str] = field(default_factory=lambda: ["rotated_ellipse"])
    # Per-shape alpha search. None → every shape uses ALPHA_FIXED (128). A list lets each
    # shape pick the opacity that fits best — high alpha for crisp black lines / dark eye
    # detail, low alpha for smooth gradients. The chosen alpha is written to the JSON color.
    alpha_levels: list[int] | None = None
    # Edge-weighted loss. 0 → uniform loss. >0 weights error by target edge magnitude so the
    # search prioritizes high-contrast detail (eyes, lineart, boundaries) over flat regions.
    # Typical useful range 1-4; weight map is 1 + edge_strength * normalized_edge.
    edge_strength: float = 0.0
    # Posterize the target to this many color levels per channel before fitting (0 or >=256
    # = off). Flattens smooth gradients into flat bands → shapes fit cleaner, edges/lineart
    # come in crisper. This is what the geometrize-based tools do (posterizeLevels 20-100).
    posterize_levels: int = 0
    # Bounding-box-local random scoring (ellipse-only, gradient mode). Scores each random
    # candidate over its own crop instead of the full canvas → ~10x+ cheaper per sample, so
    # random_samples can go far higher (geometrize-style volume) for the same VRAM. Ignored
    # for non-ellipse kinds or hillclimb mode.
    bbox_local: bool = False
    bbox_crop_max: int = 256   # cap on crop half-size (px); bounds memory for huge shapes
    # Joint-polish: after greedy, gradient-optimize ALL shapes' geometry+color+alpha against
    # the full target for this many steps (0 = off). Breaks greedy myopia — early shapes get
    # un-stuck. Ellipse-only. Cost is one optimization run, not a per-shape search.
    joint_polish_steps: int = 0
    joint_polish_lr: float = 1.5
    # polish_purity_penalty: when > 0, joint_polish adds a per-shape MSE-equivalent
    # 'spillover' loss = mass_i × variance_under_mask_i, summed over shapes, normalized
    # by canvas pixel count. This is exactly the MSE contribution shape i would make if
    # it painted with its optimal mean color, so the penalty has the SAME UNITS as the
    # main MSE loss above.
    #
    # Tuning intuition:
    #   = 1.0  → lazy multi-color paint penalty equals its MSE saving (break-even with omission)
    #   < 1.0  → painting preferred, but homogeneous coverage strongly encouraged
    #   > 1.0  → no-paint strictly preferred to lazy paint over the same multi-color region
    #
    # Without this, plain MSE actively REWARDS painting a big ellipse across multi-color
    # content with an averaged color (a 5x5 yellow star on dark loses ~23% MSE to a lazy
    # 10x10 averaged ellipse vs no-shape). That's why polish smeared sparkles into blobs
    # on celeste-with-stars. The mass-weighted form fixes the original PR #19 formulation,
    # which used per-shape AVERAGE variance — mass-independent — so huge lazy ellipses paid
    # the same penalty as tiny ones while their MSE gradient was 1000x bigger. The penalty
    # now scales with what the shape actually paints.
    #
    # Default 0.0 = legacy MSE-only polish. Production starting point: 1.0.
    polish_purity_penalty: float = 0.0
    # polish_freeze_geometry: when True (DEFAULT), joint_polish runs Adam over (rgb, alpha)
    # ONLY — x, y, rx, ry, angle from greedy are preserved bit-identically. Closed-form RGB
    # snap-back (when lock_alpha=True) still applies. This is the production polish mode:
    # Adam-over-geometry is gradient-greedy on a parameter space ellipses don't naturally
    # live in and consistently finds degenerate exploits (size_anchor: inflated;
    # purity_penalty: collapse to rx=1). Freezing geometry eliminates the entire failure
    # mode while keeping polish's actual win — fixing Adam-overshot colors via snap-back.
    # Verified at 1000 shapes (1000/1000 geom frozen, engine↔upstream parity 0.000).
    # Set False for the legacy joint-geom-and-color polish (preserved as an escape hatch
    # for future investigations into structured shape refinement). polish_purity_penalty
    # is a no-op when freeze_geometry=True (it's a geometry-affecting term).
    polish_freeze_geometry: bool = True
    # refill_dead_shapes: after greedy + joint_polish + snap-back, identify shapes whose
    # visible-pixel count under z-order is below refill_min_visible_px (fully occluded by
    # later commits = dead weight in the JSON). Drop them and run a mini-greedy targeting
    # the current residual to commit the same number of NEW shapes back, then snap-back
    # over the full set. User asked for num_shapes; gets num_shapes LIVE shapes instead
    # of num_shapes commits with ~5% dead weight. Default True — strict quality improvement
    # at the cost of one extra mini-greedy pass.
    refill_dead_shapes: bool = True
    refill_min_visible_px: int = 5
    # lock_alpha: HARD SYSTEM CONSTRAINT, NOT A USER PREFERENCE.
    # The Forza injector writes alpha=255 to every layer at inject time — there's no way
    # to ship a non-opaque shape into the game's vinyl group without modifying the EXE
    # itself. A JSON generated with soft alpha (alpha_levels=[96,160,255]) renders one
    # way in the notebook's engine preview and ANOTHER way in-game, breaking the entire
    # promise that "what you see in the preview is what the game shows."
    # Therefore False is not a supported value — it's a footgun. run_gpu validates this
    # and raises ValueError if anyone passes False. Default stays True; tests that
    # exercise the engine's old soft-alpha code path (test_joint_polish_lock.py) call
    # the internal joint_polish() function directly with lock_alpha=False, bypassing the
    # GPUConfig gate, because that function's behavior is still useful to characterize
    # for diagnostic purposes — but the production pipeline cannot accept False.
    lock_alpha: bool = True
    # prune_threshold: drop a shape if removing it changes full-canvas RMS by less than this
    # many uint8 units. 0.0 = no pruning. Currently unused by run_gpu; kept for forward-compat
    # with reprocess flows.
    prune_threshold: float = 0.5
    # Binarize the sticker alpha mask at this level (0 = off, keep soft alpha). Kills the white
    # halo from anti-aliased edges on stickers exported over a light background. 128 is the
    # midpoint; raise toward 160-200 if a halo persists.
    alpha_threshold: int = 0
    # hill-climb knobs (refine_mode == "hillclimb")
    mutation_rounds: int = 4
    mutations_per_round: int = 128
    # gradient knobs (refine_mode == "gradient")
    grad_starts: int = 16
    grad_steps: int = 50
    grad_lr: float = 2.0


def _random_params(K: int, w: int, h: int, gen: torch.Generator) -> torch.Tensor:
    """Sample K random ellipses on the CPU generator (deterministic), return CPU tensor."""
    out = torch.empty((K, 5), dtype=DTYPE)
    out[:, 0] = torch.rand(K, generator=gen) * w
    out[:, 1] = torch.rand(K, generator=gen) * h
    out[:, 2] = 1.0 + torch.rand(K, generator=gen) * (w / 8)
    out[:, 3] = 1.0 + torch.rand(K, generator=gen) * (h / 8)
    out[:, 4] = torch.rand(K, generator=gen) * 180.0
    return out


def _mutate(best: torch.Tensor, K: int, w: int, h: int, gen: torch.Generator) -> torch.Tensor:
    """K children of `best`, with the 4 CPU mutation modes evenly distributed.

    Splits K into 4 groups, one per single-axis (or fine-axis) mode, so a batch round
    explores each axis independently — this is what lets the search reach extreme aspect
    ratios (rx >> ry) that all-axis-coarse mutation can't find.
      0: translate (x, y)        2: rotate (angle)
      1: resize (rx, ry)         3: fine xy + angle
    """
    base = best.view(1, 5).expand(K, 5).contiguous()
    noise = torch.zeros((K, 5), dtype=DTYPE)
    per = max(1, K // 4)

    s = slice(0, per); n = s.stop - s.start
    noise[s, 0] = torch.randn(n, generator=gen) * _MUT_SIGMA_XY_COARSE
    noise[s, 1] = torch.randn(n, generator=gen) * _MUT_SIGMA_XY_COARSE

    s = slice(per, 2 * per); n = s.stop - s.start
    noise[s, 2] = torch.randn(n, generator=gen) * _MUT_SIGMA_R_COARSE
    noise[s, 3] = torch.randn(n, generator=gen) * _MUT_SIGMA_R_COARSE

    s = slice(2 * per, 3 * per); n = s.stop - s.start
    noise[s, 4] = torch.randn(n, generator=gen) * _MUT_SIGMA_ANGLE

    s = slice(3 * per, K); n = s.stop - s.start
    if n > 0:
        noise[s, 0] = torch.randn(n, generator=gen) * _MUT_SIGMA_XY_FINE
        noise[s, 1] = torch.randn(n, generator=gen) * _MUT_SIGMA_XY_FINE
        noise[s, 4] = torch.randn(n, generator=gen) * _MUT_SIGMA_ANGLE_FINE

    out = base + noise
    out[:, 0].clamp_(0.0, w - 1)
    out[:, 1].clamp_(0.0, h - 1)
    out[:, 2].clamp_(1.0, float(w))
    out[:, 3].clamp_(1.0, float(h))
    out[:, 4] = out[:, 4] % 180.0
    return out


def _composite_one(kind: ShapeKind, canvas_u8: torch.Tensor, params: torch.Tensor,
                   color_u8: torch.Tensor, h: int, w: int,
                   alpha_mask_f: torch.Tensor | None, alpha: int = ALPHA_FIXED) -> torch.Tensor:
    """Composite a single shape (of the given kind) with RGB color + `alpha` onto canvas.
    If alpha_mask_f is provided, the shape's effective mask is `mask * alpha_mask_f` so
    paint never lands in transparent areas (the grey canvas init stays visible there).
    """
    mask = kind.rasterize_hard(params.view(1, -1), h, w)[0]   # (H, W)
    if alpha_mask_f is not None:
        mask = mask * alpha_mask_f
    a = float(alpha) / 255.0
    cur = canvas_u8.to(DTYPE)
    src = color_u8.to(DTYPE).view(1, 1, 3)
    m = mask.unsqueeze(-1)
    blended = m * (a * src + (1.0 - a) * cur) + (1.0 - m) * cur
    return blended.clamp(0.0, 255.0).round().to(torch.uint8)


# --- Gradient-based refinement (differentiable rasterizer + Adam) ---

def _posterize(target_rgb: np.ndarray, levels: int) -> np.ndarray:
    """Quantize an (H, W, 3) uint8 image to `levels` evenly-spaced values per channel.
    levels<2 or >=256 returns the image unchanged. Flattens gradients into bands so the
    greedy shape fit produces cleaner flat regions and crisper edges (geometrize does this).
    """
    if levels < 2 or levels >= 256:
        return target_rgb
    f = target_rgb.astype(np.float32) / 255.0
    q = np.round(f * (levels - 1)) / (levels - 1) * 255.0
    return q.round().clip(0, 255).astype(np.uint8)


def _edge_weight_map(target_u8: torch.Tensor, strength: float) -> torch.Tensor:
    """(H, W, 3) uint8 target → (H, W) per-pixel loss weight = 1 + strength * norm_edge.

    CHROMA-AWARE: the gradient is computed per RGB channel and combined as the L2 over both
    spatial directions AND channels — `sqrt(sum_c(gx_c^2 + gy_c^2))`. This catches *color*
    edges that a luminance-only (grayscale-mean) gradient misses: a feature that shifts hue
    but not brightness (e.g. a faint tattoo on skin) cancels in the channel mean but shows
    up strongly per channel. It still fully captures luminance edges (those appear in all
    channels). Normalized by the 99th percentile; high near eyes/lineart/colored detail.
    """
    t = target_u8.to(DTYPE)                                      # (H, W, 3)
    gx = torch.zeros_like(t); gy = torch.zeros_like(t)
    gx[:, 1:-1, :] = (t[:, 2:, :] - t[:, :-2, :]) * 0.5
    gy[1:-1, :, :] = (t[2:, :, :] - t[:-2, :, :]) * 0.5
    edge = torch.sqrt((gx * gx + gy * gy).sum(dim=2))            # (H, W) over channels
    flat = edge.flatten()
    # 99th percentile via kthvalue (quantile can be unsupported / slow on some backends)
    k = max(1, int(0.99 * flat.numel()))
    norm = torch.kthvalue(flat, k).values.clamp(min=1e-3)
    edge_norm = (edge / norm).clamp(0.0, 1.0)
    return 1.0 + float(strength) * edge_norm


def _optimal_color_batch(masks, cur_f, tgt_f, alpha_mask_f, a):
    """Closed-form optimal RGB per candidate (M, 3), float 0..255. Caller detaches."""
    if alpha_mask_f is not None:
        eff = masks * alpha_mask_f.unsqueeze(0)
    else:
        eff = masks
    m = eff.unsqueeze(-1)
    weight = eff.sum(dim=(1, 2)).clamp(min=1e-3)
    sum_mt = (m * tgt_f).sum(dim=(1, 2))
    sum_mc = (m * cur_f).sum(dim=(1, 2))
    return ((sum_mt - (1.0 - a) * sum_mc) / (a * weight.unsqueeze(-1))).clamp(0.0, 255.0)


def _refine_gradient(kind: ShapeKind, init_params, canvas_u8, target_u8, alpha_t,
                     alpha_mask_f, h, w, steps, lr, alpha_levels=None, edge_weight=None):
    """Multi-start Adam refinement for one shape kind. init_params is (M, P). Returns the
    single best (params (P,), color_u8 (3,), alpha int, score float).

    Geometry is refined at ALPHA_FIXED (alpha mostly affects intensity, not where a shape
    should sit). Color is recomputed closed-form each step and detached (alternating
    optimization). The final selection pools init + refined and picks the global best via
    the real hard-mask, sticker-aware scorer WITH alpha search — so the committed shape
    gets the opacity that fits best. Gradient descent can never do worse than the random
    seed (the seed candidates are in the final pool).
    """
    a = ALPHA_FIXED / 255.0
    cur_f = canvas_u8.to(DTYPE)
    tgt_f = target_u8.to(DTYPE)
    M = init_params.shape[0]

    # Per-pixel loss weight = (alpha gate, if sticker) × (edge emphasis, if enabled).
    lw_2d = None
    if alpha_t is not None:
        lw_2d = (alpha_t > 0).to(DTYPE)
    if edge_weight is not None:
        lw_2d = edge_weight if lw_2d is None else lw_2d * edge_weight
    if lw_2d is not None:
        lw4 = lw_2d.view(1, h, w, 1)
        denom = (lw_2d.sum() * 3.0).clamp(min=1.0)

    p = init_params.clone().detach().requires_grad_(True)
    opt = torch.optim.Adam([p], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        masks = kind.rasterize_soft(p, h, w)
        color = _optimal_color_batch(masks, cur_f, tgt_f, alpha_mask_f, a).detach()
        src = color.view(M, 1, 1, 3)
        if alpha_mask_f is not None:
            m_paint = (masks * alpha_mask_f.unsqueeze(0)).unsqueeze(-1)
        else:
            m_paint = masks.unsqueeze(-1)
        blended = m_paint * (a * src + (1.0 - a) * cur_f) + (1.0 - m_paint) * cur_f
        diff = blended - tgt_f
        if lw_2d is not None:
            per = (diff * diff * lw4).sum(dim=(1, 2, 3)) / denom
        else:
            per = (diff * diff).mean(dim=(1, 2, 3))
        if alpha_mask_f is not None:
            bleed = (masks * (1.0 - alpha_mask_f.unsqueeze(0))).sum(dim=(1, 2))
            per = per + STICKER_GRAD_PENALTY * bleed
        per.sum().backward()
        opt.step()
        with torch.no_grad():
            kind.clamp_(p, w, h)

    refined = p.detach()
    candidates = torch.cat([init_params, refined], dim=0)
    masks = kind.rasterize_hard(candidates, h, w)
    scores, colors, alphas = score_batch(masks, canvas_u8, target_u8, alpha_mask=alpha_t,
                                         alpha_levels=alpha_levels, edge_weight=edge_weight)
    best = int(scores.argmin().cpu().item())
    return (candidates[best], colors[best], int(alphas[best].cpu().item()),
            float(scores[best].cpu().item()))


def run_gpu(
    target_rgb: np.ndarray,
    cfg: GPUConfig,
    alpha_mask: np.ndarray | None = None,
    progress_every: int = 0,
    checkpoint_cb=None,
    checkpoint_every: int = 0,
) -> tuple[list[dict], np.ndarray]:
    """Public entry point — wraps `_run_gpu_inner` with CUDA OOM recovery.

    On `torch.cuda.OutOfMemoryError`: drops CUDA cache so the runtime sees
    free memory on the next attempt, then raises `RuntimeError` with an
    actionable recipe naming the specific knobs to lower (MAX_RESOLUTION,
    RANDOM_SAMPLES). Without this wrapper users get a raw torch traceback
    that doesn't tell them what to change — and the cache holds the OOM
    state so the next attempt without restart STILL OOMs even on identical
    parameters that would have fit cold.

    Critical for the consumer-GPU notebook variants where the VRAM probe
    can underestimate peak usage when FH6 is running concurrently and its
    VRAM consumption fluctuates mid-shape-gen.
    """
    try:
        return _run_gpu_inner(target_rgb, cfg, alpha_mask=alpha_mask,
                              progress_every=progress_every,
                              checkpoint_cb=checkpoint_cb,
                              checkpoint_every=checkpoint_every)
    except torch.cuda.OutOfMemoryError as e:
        # Drop the cached allocator state so a follow-up attempt with
        # lower settings doesn't inherit this run's high-water mark.
        peak_mib = 0.0
        try:
            if torch.cuda.is_available():
                peak_mib = torch.cuda.max_memory_allocated() / (1 << 20)
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass
        h, w = target_rgb.shape[:2]
        # Concrete halving recommendations based on what the user set.
        rec_max = max(240, int(max(h, w) // 2))
        rec_samples = max(1024, cfg.random_samples // 2)
        raise RuntimeError(
            f"CUDA out of memory during shape-gen at {w}x{h} canvas. "
            f"Peak VRAM observed: ~{peak_mib / 1024:.1f} GiB.\n\n"
            f"Recovery — re-run the Configure cell with EITHER:\n"
            f"  • MAX_RESOLUTION ≤ {rec_max} (halve the canvas long side), OR\n"
            f"  • RANDOM_SAMPLES ≤ {rec_samples} (halve the candidate batch)\n"
            f"  • Both for max safety\n\n"
            f"Then re-run from the Resolution Planner cell (it'll re-verify "
            f"the new settings fit). If you have FH6 open and it's using a "
            f"lot of VRAM, closing it briefly frees ~4–6 GiB for the run.\n\n"
            f"Original error: {e}"
        ) from e


def _run_gpu_inner(
    target_rgb: np.ndarray,
    cfg: GPUConfig,
    alpha_mask: np.ndarray | None = None,
    progress_every: int = 0,
    checkpoint_cb=None,
    checkpoint_every: int = 0,
) -> tuple[list[dict], np.ndarray]:
    """Run the GPU shape-gen loop. Returns (shapes_as_json_dicts, final_canvas_u8_numpy).

    If `checkpoint_cb` and `checkpoint_every > 0`, the greedy phase calls
    `checkpoint_cb(shape_idx, list_of_shapes_so_far)` every `checkpoint_every` committed
    shapes — so a caller can persist recoverable progress (e.g. to Google Drive) and survive
    a session reset mid-run. The callback gets a cheap copy of the shapes list (no canvas).

    If `alpha_mask` is provided (H×W uint8, 0=transparent, 255=opaque), sticker mode is
    active: target RGB is zeroed in transparent areas, canvas inits to grey (40), the
    scorer rejects candidates that bleed past the silhouette, and paint is clipped to
    the opaque region. Caller must match the alpha_mask's H/W to target_rgb's H/W.

    If `progress_every > 0`, prints a progress line every that many committed shapes with
    rate, ETA, and current best RMS — useful for long Colab runs.
    """
    # HARD SYSTEM CONSTRAINT — see GPUConfig.lock_alpha comment. The Forza injector
    # writes alpha=255 unconditionally; a JSON generated with soft alpha is not
    # representable in-game. Reject the invalid state at the API boundary so users
    # can't silently ship a broken JSON.
    if not cfg.lock_alpha:
        raise ValueError(
            "GPUConfig.lock_alpha must be True. The Forza injector writes alpha=255 "
            "to every layer at inject time — there's no path to ship a soft-alpha JSON "
            "into the game's vinyl group. A False value would generate a JSON whose "
            "engine preview shows soft alpha but whose in-game render is fully opaque, "
            "breaking the preview-matches-game contract. (If you're benchmarking the "
            "old soft-alpha code path for diagnostic reasons, call joint_polish() "
            "directly with lock_alpha=False instead of going through run_gpu.)"
        )
    if target_rgb.ndim != 3 or target_rgb.shape[2] != 3 or target_rgb.dtype != np.uint8:
        raise ValueError("target_rgb must be (H, W, 3) uint8")
    if alpha_mask is not None:
        if alpha_mask.ndim != 2 or alpha_mask.dtype != np.uint8:
            raise ValueError("alpha_mask must be (H, W) uint8 or None")
        if alpha_mask.shape != target_rgb.shape[:2]:
            raise ValueError(
                f"alpha_mask shape {alpha_mask.shape} != target {target_rgb.shape[:2]}"
            )
        # Alpha threshold: binarize the mask at this level (0 = off, keep soft alpha). Stickers
        # exported over a light background have an anti-aliased edge whose semi-transparent
        # pixels carry that light RGB; counting them as opaque paints a bright HALO around the
        # silhouette on the grey substrate. Thresholding drops the soft fringe → clean hard
        # silhouette (which is what FH6 renders anyway). Raise toward 160-200 if a halo persists.
        if cfg.alpha_threshold > 0:
            alpha_mask = np.where(alpha_mask >= cfg.alpha_threshold, 255, 0).astype(np.uint8)

    if cfg.lock_alpha:
        # Lock every shape to alpha=255: the FH injector forces binary alpha at inject time,
        # so optimizing soft alpha produces JSONs whose engine PNG diverges from the game render.
        cfg = replace(cfg, alpha_levels=[255])

    device = get_device()
    h, w = target_rgb.shape[:2]

    # Posterize the target (color quantization) before fitting — cleaner flat bands, crisper
    # edges. Done on the raw target so both the scored target and the canvas mean see it.
    if cfg.posterize_levels:
        target_rgb = _posterize(target_rgb, cfg.posterize_levels)

    # Resolve candidate shape kinds. Gradient mode tries all; hillclimb is ellipse-only
    # (no per-kind mutation is defined for triangle/rectangle).
    kinds = [KINDS[t] for t in cfg.shape_types]
    if cfg.refine_mode == "hillclimb":
        kinds = [KINDS["rotated_ellipse"]]

    # bbox-local random scoring applies only to the single-ellipse gradient path.
    use_bbox = (cfg.bbox_local and cfg.refine_mode == "gradient"
                and len(kinds) == 1 and kinds[0].name == "rotated_ellipse")

    if alpha_mask is not None:
        # Fill out-of-silhouette target pixels with the canvas substrate color (grey 40).
        # Rationale: in-game and CPU renderers paint every shape's full ellipse unclipped —
        # they have no source alpha mask to clip against. To make the optimizer's view of the
        # world match the game's, we DON'T clip shapes in the renderer. But then we need a
        # different mechanism to discourage paint from spilling past the silhouette. The trick:
        # if target == canvas_init outside the silhouette, then "no paint there" produces zero
        # loss, and "paint that spills there" produces positive loss — naturally penalizing
        # spill via plain MSE. Previously target was zeroed (RGB=0) outside the silhouette and
        # the loss was lw-masked to ignore those pixels; the optimizer never felt spill and
        # joint_polish would silently let shapes drift across the boundary, which the engine
        # then hid by clipping at render time. That clipped preview diverged from the unclipped
        # game render. Aligning target with substrate fixes the loss signal so JSON quality
        # matches what the game will actually display.
        opaque_mask3 = (alpha_mask > 0)[:, :, None].astype(np.uint8)
        substrate = np.full_like(target_rgb, 40)
        target_clean = np.where(opaque_mask3 > 0, target_rgb, substrate)
    else:
        target_clean = target_rgb

    target = torch.from_numpy(target_clean).to(device)
    alpha_t: torch.Tensor | None = None
    alpha_mask_f: torch.Tensor | None = None
    if alpha_mask is not None:
        alpha_t = torch.from_numpy(alpha_mask).to(device)
        alpha_mask_f = alpha_t.to(DTYPE) / 255.0

    # Edge-weight map (constant across the run): emphasizes eyes/lineart/boundaries.
    edge_weight = _edge_weight_map(target, cfg.edge_strength) if cfg.edge_strength > 0 else None

    if alpha_mask is not None:
        canvas = torch.full((h, w, 3), 40, dtype=torch.uint8, device=device)
    else:
        mean = target.to(DTYPE).reshape(-1, 3).mean(dim=0).round().clamp(0, 255).to(torch.uint8)
        canvas = mean.view(1, 1, 3).expand(h, w, 3).contiguous().clone()

    seed = cfg.seed or (int(time.time() * 1000) & 0x7FFFFFFF)
    gen = torch.Generator(device="cpu").manual_seed(seed)

    shapes: list[dict] = []
    consecutive_skips = 0
    shape_idx = 0
    t_start = time.perf_counter()
    per_kind = max(1, cfg.random_samples // len(kinds))
    while shape_idx < cfg.num_shapes:
        # 1) Random search across all candidate kinds; keep each kind's best seed.
        #    score_batch is type-agnostic (operates on the rasterized mask), so all kinds
        #    are scored the same way and compared directly.
        best_kind = None
        best_kind_params = None   # (per_kind, P) device tensor of the winning kind
        best_kind_scores = None   # per-candidate scores (RMS, or bbox delta) for top-M
        best_score = float("inf")
        best_color = None
        best_alpha = ALPHA_FIXED
        if use_bbox:
            # bbox-local: score each random ellipse over its own crop (cheap → high volume).
            # `scores` here are incremental SSE deltas (lower=better); argmin/top-M still
            # select the globally-best candidate. The committed RMS is recomputed full-canvas
            # by the gradient refiner below.
            ekind = kinds[0]
            params = ekind.init(cfg.random_samples, w, h, gen).to(device)
            scores, colors, alphas = crop_score_ellipse_batch(
                params, canvas, target, alpha_t=alpha_t, edge_weight=edge_weight,
                alpha_levels=cfg.alpha_levels, max_crop_radius=cfg.bbox_crop_max)
            idx_t = scores.argmin()
            best_score = float(scores[idx_t].cpu().item())
            if best_score != float("inf"):
                best_kind = ekind
                best_kind_params = params
                best_kind_scores = scores
                bi = int(idx_t.cpu().item())
                best_color = colors[bi].clone()
                best_alpha = int(alphas[bi].cpu().item())
                best_params = params[bi].clone()
        else:
            for kind in kinds:
                params = kind.init(per_kind, w, h, gen).to(device)
                masks = kind.rasterize_hard(params, h, w)
                scores, colors, alphas = score_batch(masks, canvas, target, alpha_mask=alpha_t,
                                                     alpha_levels=cfg.alpha_levels,
                                                     edge_weight=edge_weight)
                idx_t = scores.argmin()
                val = float(scores[idx_t].cpu().item())
                if val < best_score:
                    best_score = val
                    best_kind = kind
                    best_kind_params = params
                    best_kind_scores = scores
                    bi = int(idx_t.cpu().item())
                    best_color = colors[bi].clone()
                    best_alpha = int(alphas[bi].cpu().item())
                    best_params = params[bi].clone()
                del masks
                if kind is not best_kind:
                    del params, scores, colors, alphas

        if best_score == float("inf"):
            consecutive_skips += 1
            if consecutive_skips >= _MAX_CONSECUTIVE_SKIPS:
                break
            continue

        if cfg.refine_mode == "gradient":
            # Multi-start within the winning kind: top-M of its candidates.
            M = min(cfg.grad_starts, best_kind_params.shape[0])
            top_idx = torch.topk(best_kind_scores, M, largest=False).indices
            init = best_kind_params[top_idx].clone()
            best_params, best_color, best_alpha, best_score = _refine_gradient(
                best_kind, init, canvas, target, alpha_t, alpha_mask_f, h, w,
                cfg.grad_steps, cfg.grad_lr, alpha_levels=cfg.alpha_levels,
                edge_weight=edge_weight,
            )
            if best_score == float("inf"):
                consecutive_skips += 1
                if consecutive_skips >= _MAX_CONSECUTIVE_SKIPS:
                    break
                continue
        else:
            # Hill-climb (rotated_ellipse only — kinds was forced to ellipse above).
            for _r in range(cfg.mutation_rounds):
                mut_cpu = _mutate(best_params.cpu(), cfg.mutations_per_round, w, h, gen)
                mut = mut_cpu.to(device)
                mut_masks = rasterize_rotated_ellipses(mut, h, w)
                mut_scores, mut_colors, mut_alphas = score_batch(
                    mut_masks, canvas, target, alpha_mask=alpha_t, alpha_levels=cfg.alpha_levels,
                    edge_weight=edge_weight)
                local_best_t = mut_scores.argmin()
                local_best_score = float(mut_scores[local_best_t].cpu().item())
                if local_best_score < best_score:
                    best_score = local_best_score
                    local_best = int(local_best_t.cpu().item())
                    best_params = mut[local_best].clone()
                    best_color = mut_colors[local_best].clone()
                    best_alpha = int(mut_alphas[local_best].cpu().item())
                del mut_masks, mut_scores, mut_colors, mut_alphas, mut, mut_cpu

        # Commit
        canvas = _composite_one(best_kind, canvas, best_params, best_color, h, w,
                                alpha_mask_f, best_alpha)
        c = best_color.cpu().tolist()
        color = [int(c[0]), int(c[1]), int(c[2]), int(best_alpha)]
        shapes.append(best_kind.to_json(best_params.cpu().tolist(), color))
        consecutive_skips = 0
        shape_idx += 1

        if checkpoint_cb and checkpoint_every and shape_idx % checkpoint_every == 0:
            checkpoint_cb(shape_idx, list(shapes))

        if progress_every and shape_idx % progress_every == 0:
            elapsed = time.perf_counter() - t_start
            rate = shape_idx / elapsed if elapsed > 0 else 0.0
            eta = (cfg.num_shapes - shape_idx) / rate if rate > 0 else 0.0
            print(f"  {shape_idx}/{cfg.num_shapes} shapes  "
                  f"({rate:.1f}/s, ETA {eta:.0f}s, RMS {best_score:.2f})")

    # Joint-polish: globally refine all committed shapes (ellipse-only) against the target.
    if cfg.joint_polish_steps > 0 and shapes and all(s["type"] == "rotated_ellipse" for s in shapes):
        if progress_every:
            print(f"  joint-polishing {len(shapes)} shapes for {cfg.joint_polish_steps} steps...")
        shapes, canvas_np = joint_polish(
            shapes, target, alpha_t, alpha_mask_f, edge_weight, canvas,
            h, w, cfg.joint_polish_steps, lr=cfg.joint_polish_lr,
            progress=bool(progress_every), lock_alpha=cfg.lock_alpha,
            purity_penalty=cfg.polish_purity_penalty,
            freeze_geometry=cfg.polish_freeze_geometry,
        )
        if cfg.refill_dead_shapes and shapes and all(s["type"] == "rotated_ellipse" for s in shapes):
            n_before = len(shapes)
            shapes, canvas_np = clean_and_refill(
                shapes, target, canvas, alpha_t, alpha_mask_f, edge_weight,
                cfg, h, w, gen,
            )
            if progress_every:
                n_refilled = len(shapes) - n_before if len(shapes) > n_before else 0
                print(f"  refill: final {len(shapes)} live shapes "
                      f"(detected dead + refilled {n_refilled} this pass)")
        return shapes, canvas_np

    if cfg.refill_dead_shapes and shapes and all(s["type"] == "rotated_ellipse" for s in shapes):
        # No-polish path: refill still runs because dead-shape detection works on greedy
        # output directly (snap-back only matters if there was a polish to snap from; the
        # refill path's internal snap-back handles either case). Multi-shape eval presets
        # skip refill entirely — the mini-greedy commits ellipses only.
        n_before = len(shapes)
        shapes, canvas_np = clean_and_refill(
            shapes, target, canvas, alpha_t, alpha_mask_f, edge_weight,
            cfg, h, w, gen,
        )
        if progress_every:
            n_refilled = len(shapes) - n_before if len(shapes) > n_before else 0
            print(f"  refill: final {len(shapes)} live shapes "
                  f"(detected dead + refilled {n_refilled} this pass)")
        return shapes, canvas_np
    return shapes, canvas.cpu().numpy()

DEVICE = get_device()
print(f"Engine loaded on {DEVICE}. Ready for image upload.")

## 2.5 VRAM autopicker — **am I in the right notebook?**

Probes free VRAM and whether FH6 is running locally. Prints a recommended notebook variant based on what your environment can support. If you're already in the right one, continue. If not, open the recommended one instead — same workflow, settings tuned for your card.

In [ ]:
# --- VRAM autopicker — what preset should you use? ---
# Probes:
#   (a) Free VRAM via torch.cuda.mem_get_info() — what's actually available
#       right now, not just total card capacity.
#   (b) Whether forzahorizon6.exe is running (local-Windows only — Colab
#       always reports False since the game can't run there). On Windows
#       with FH6 open, the game holds 4-6 GiB of VRAM, so shape-gen has to
#       share. On Colab the GPU is dedicated.
# Then prints a recommendation table mapping (free_vram, fh6_running) →
# notebook preset. You manually open the recommended notebook if it differs
# from the one you're in. Future work: auto-adjust this notebook's Configure
# defaults to match the recommendation.
import torch

try:
    import psutil
    _PSUTIL_OK = True
except ImportError:
    _PSUTIL_OK = False
    print("(psutil not installed — skipping FH6 process detect. pip install psutil to enable.)")

def _fh6_running() -> bool:
    """Return True iff forzahorizon6.exe is in the current process list. False
    on non-Windows systems or when psutil is missing — Colab always returns
    False since the game can't run in the Colab runtime."""
    if not _PSUTIL_OK:
        return False
    targets = {"forzahorizon6.exe", "forzahorizon6-win64-shipping.exe"}
    for p in psutil.process_iter(["name"]):
        try:
            n = (p.info["name"] or "").lower()
            if n in targets:
                return True
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return False

def _gpu_state():
    """Returns (gpu_name, free_gib, total_gib) or (None, 0, 0) if no CUDA."""
    if not torch.cuda.is_available():
        return (None, 0.0, 0.0)
    free_b, total_b = torch.cuda.mem_get_info()
    name = torch.cuda.get_device_name(0)
    return (name, free_b / (1 << 30), total_b / (1 << 30))

_gpu_name, _free_gib, _total_gib = _gpu_state()
_fh6 = _fh6_running()

print(f"GPU:        {_gpu_name or '(no CUDA)'}")
if _gpu_name:
    print(f"VRAM:       {_free_gib:.1f} GiB free / {_total_gib:.1f} GiB total")
print(f"FH6 running: {'YES — sharing the GPU' if _fh6 else 'no (or undetectable)'}")
print()

# Decision matrix: (free_gib_floor, fh6_running) → recommended notebook name.
# Floors are lower bounds — if you have >= floor GiB free, you can use this preset.
# Tuned conservatively: ~30% headroom buffer to absorb peak overshoot beyond the
# resolution planner's estimate.
_recs = []
if _fh6:
    if _free_gib >= 12:
        _recs.append(("fap_gpu_local_consumer_1000", "1000 shapes, MAX_RES=720, FH6-coresident OK"))
    elif _free_gib >= 6:
        _recs.append(("fap_gpu_local_consumer_700", "700 shapes, MAX_RES=600"))
    elif _free_gib >= 3:
        _recs.append(("fap_gpu_local_consumer_400", "400 shapes, MAX_RES=480"))
    else:
        _recs.append((None, f"<3 GiB free with FH6 running — close the game first OR open a fap_gpu_colab_*.ipynb on Colab"))
else:
    if _free_gib >= 24:
        _recs.append(("fap_gpu_colab_highres_3000 / fap_gpu_local_overnight_3000", "3000 shapes, MAX_RES=1600, full quality"))
    elif _free_gib >= 12:
        _recs.append(("fap_gpu_colab_medium_1000 / fap_gpu_local_overnight_1000", "1000 shapes, MAX_RES=1000"))
    elif _free_gib >= 6:
        _recs.append(("fap_gpu_colab_headshots_700", "700 shapes, MAX_RES=900"))
    elif _free_gib > 0:
        _recs.append(("fap_gpu_colab_lineart_400", "400 shapes, MAX_RES=720"))
    else:
        _recs.append((None, "no CUDA available — runtime needs Hardware Accelerator: GPU"))

print("RECOMMENDATION based on your environment:")
for nb, desc in _recs:
    if nb:
        print(f"  → {nb}.ipynb  ({desc})")
    else:
        print(f"  → {desc}")
print()
print("If you're not in the recommended notebook, open it instead of continuing here.")
print("If you ARE in the recommended notebook (or accept the risk of a heavier one),")
print("continue to the next cell.")


## 3. 🧹 Cleanup (run between attempts)

Use this if a previous run OOM'd or you want to start fresh with different parameters. If `allocated` stays >1 GB after this cell, restart the runtime (Runtime → Restart runtime).

In [ ]:
# --- Cleanup: free CUDA memory between runs ---
# Run this cell after every failed/successful run when you want to change parameters
# and re-run. Clears the run artifacts AND the IPython output history that pins them
# (the most common reason a "fresh" run still OOMs).
import sys, gc, torch

# 1) Clear Jupyter's last-traceback (holds frame locals = live tensors after a crash)
sys.last_traceback = None
sys.last_value = None
sys.last_type = None

# 2) Drop user-level variables that hold GPU tensors or run outputs
for _name in (
    # Run outputs
    "shapes_json", "final_canvas", "cfg",
    # Preprocessed inputs (CPU but worth clearing to drop the IPython _N hold)
    "target_rgb", "alpha_mask", "h", "w", "mode_tag",
    # Anything that might have leaked from a previous engine call
    "target", "alpha_t", "alpha_mask_f", "canvas",
    "masks", "scores", "colors", "params", "params_cpu",
    "mut_masks", "mut_scores", "mut_colors", "mut", "mut_cpu",
    "best_params", "best_color",
):
    globals().pop(_name, None)

# 3) Clear IPython output history (Out[i], _, __, ___, _oh) — biggest hidden leak
try:
    ip = get_ipython()
    ip.user_ns.pop("_", None)
    ip.user_ns.pop("__", None)
    ip.user_ns.pop("___", None)
    if "_oh" in ip.user_ns:
        ip.user_ns["_oh"].clear()
    if "Out" in ip.user_ns:
        ip.user_ns["Out"].clear()
except (NameError, AttributeError):
    pass

# 4) Force GC and release the allocator pool
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"after cleanup: allocated={alloc:.2f} GB, reserved={reserved:.2f} GB")
    if alloc > 1.0:
        print("WARNING: >1 GB still live. Restart the runtime if you need a fully clean slate.")
else:
    print("(no CUDA — nothing to free)")


## 4. Mount Google Drive (persistent output)

**This is the key cell for not losing results.** Outputs save straight to Drive the instant generation finishes, plus checkpoints during the run — so a session reset / disconnect can't lose your JSON + PNG. You'll be prompted to authorize Drive access.

In [ ]:
# --- Mount Google Drive (persistent output; survives session resets) ---
# The Run cell writes the final JSON + PNG here the instant generation finishes, plus
# recoverable checkpoints during the run. Even if the Colab session resets afterward, your
# output is already on Drive — no more losing results to a disconnect.
USE_DRIVE    = True            # False → save to /content only (lost on reset)
DRIVE_FOLDER = "forza_abyss_painter_output"    # subfolder under MyDrive/

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
else:
    OUTPUT_DIR = "/content/forza_abyss_painter_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs -> {OUTPUT_DIR}")
print("Per image, results land in <OUTPUT_DIR>/<image_stem>/  (final + numbered checkpoints).")


## 5. Upload an image

In [ ]:
# --- Upload an image ---
from google.colab import files
import io

print("Pick a PNG or JPG to convert into Forza vinyl shapes.")
_uploaded = files.upload()
if not _uploaded:
    raise SystemExit("No file uploaded.")

# Take the first uploaded file
_first_name, _first_bytes = next(iter(_uploaded.items()))
SOURCE_IMAGE_NAME = _first_name
SOURCE_IMAGE_BYTES = _first_bytes
_img_preview = Image.open(io.BytesIO(SOURCE_IMAGE_BYTES))
_sw, _sh = _img_preview.size
print(f"Loaded {SOURCE_IMAGE_NAME}: {_sw}x{_sh}, mode={_img_preview.mode}")
print("Set your knobs next, then the Resolution Planner will help you choose MAX_RESOLUTION.")
_img_preview


## 6. Posterize preview — **pick POSTERIZE_LEVELS**

Shows your image at 4 posterize levels with a ★ recommendation (the lowest level that won't wash out this image's fine detail). Decide your value here, then set it in the Configure cell.

In [ ]:
# --- Posterize preview: pick POSTERIZE_LEVELS for THIS image ---
# Posterize flattens color gradients into bands → cleaner fills + crisper edges, BUT too few
# levels washes out faint detail: subtle color shifts smaller than the quantization step
# vanish (this is what made the neck tattoos disappear). The panels below show 4 levels;
# the ★ recommendation is the lowest level whose step is fine enough to preserve THIS image's
# fine detail. Eyeball them, then set POSTERIZE_LEVELS in the Configure cell.
import io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

_LEVELS = [16, 24, 32, 48]

_im = Image.open(io.BytesIO(SOURCE_IMAGE_BYTES))
_im = _im.convert("RGB") if _im.mode not in ("RGB",) else _im
_w0, _h0 = _im.size
_s = 512 / max(_w0, _h0)
if _s < 1:
    _im = _im.resize((max(1, int(_w0 * _s)), max(1, int(_h0 * _s))))
_rgb = np.asarray(_im)

def _posterize_np(a, L):
    if L < 2 or L >= 256:
        return a
    f = a.astype(np.float32) / 255.0
    return (np.round(f * (L - 1)) / (L - 1) * 255.0).round().clip(0, 255).astype(np.uint8)

# Fine-detail amplitude: typical high-pass magnitude where the image carries detail. The
# posterize step (255/(L-1)) must be <= this or that detail gets quantized away.
_blur = np.asarray(_im.filter(ImageFilter.BoxBlur(2))).astype(np.float32)
_hp = np.abs(_rgb.astype(np.float32) - _blur).max(axis=2)
_det = _hp[_hp > 1.0]
_amp = float(np.percentile(_det, 40)) if _det.size > 50 else 24.0
_need = 255.0 / max(_amp, 1e-3) + 1.0
_rec = _LEVELS[-1]
for _L in _LEVELS:
    if _L >= _need:
        _rec = _L
        break

_fig, _ax = plt.subplots(1, len(_LEVELS) + 1, figsize=(3.2 * (len(_LEVELS) + 1), 3.4))
_ax[0].imshow(_rgb); _ax[0].set_title("original"); _ax[0].axis("off")
for _i, _L in enumerate(_LEVELS):
    _ax[_i + 1].imshow(_posterize_np(_rgb, _L))
    _ax[_i + 1].set_title(f"posterize {_L}" + ("  \u2605REC" if _L == _rec else ""))
    _ax[_i + 1].axis("off")
plt.tight_layout(); plt.show()

print(f"Recommended POSTERIZE_LEVELS = {_rec}")
print(f"  (fine-detail amplitude ~{_amp:.0f}/255; levels below ~{_need:.0f} start washing out")
print(f"   detail that small. Lower = cleaner banding; higher = preserve faint colored detail")
print(f"   like tattoos. 0 disables posterize entirely.)")
print("Set POSTERIZE_LEVELS in the Configure cell to your pick — the \u2605 is a starting point.")


## 7. Configure — set knobs

Preset defaults below. Set `POSTERIZE_LEVELS` from the preview above. `STICKER_MODE=True` requires an alpha channel. Re-run this cell after changing anything.

In [ ]:
# --- Knobs (MEDIUM / 1000 shapes (multi-shape EVAL — not injectable yet)) ---
# Preset tuned for this shape budget. Everything is editable; the Resolution Planner below
# validates MAX_RESOLUTION against your GPU before you launch.
REFINE_MODE        = "gradient"   # "gradient" (Adam refine) | "hillclimb" (legacy mutations)

# Candidate shape types. The FH6 injector currently renders only ellipses — triangle/
# rotated_rectangle render in PREVIEW but won't inject today (dev roadmap). Multi-shape is
# gradient-only, falls back to full-canvas scoring (lower RANDOM_SAMPLES), and skips
# joint-polish. The `shapes_*` presets enable all three for evaluating the quality ceiling.
SHAPE_TYPES = ["rotated_ellipse", "triangle", "rotated_rectangle"]

# Per-shape opacity search (crisp black lines / dark detail vs smooth gradients). The chosen
# opacity is written into each shape's JSON color. Fewer levels = faster.
ALPHA_LEVELS = [96, 160, 255]

# Edge-weighted loss: prioritize high-contrast detail (eyes, lineart) over flat regions.
# 0 = off; useful range 1-4 (higher over-focuses edges).
EDGE_STRENGTH = 3.0

# Posterize the target to N color levels before fitting (0 = off). Flattens gradients into
# bands → cleaner fills, crisper edges (the geometrize trick). 16-32 is a good range.
POSTERIZE_LEVELS = 24

# bbox-local scoring: score each random candidate over its bounding-box crop, not the full
# canvas (~10x+ cheaper) — this is what makes high RANDOM_SAMPLES affordable.
BBOX_LOCAL = True

NUM_SHAPES         = 1000   # FH6 vinyl-group limit is ~3000
RANDOM_SAMPLES     = 3072  # seed candidates/shape. forza-painter's #1 fix for
                            # "blurry" is RAISING this (they use 50k-350k). bbox-local makes
                            # it affordable; push higher to chase parity.
MAX_RESOLUTION     = 1000   # matched to NUM_SHAPES (~sqrt(N)*28 px long side).
                            # Higher wastes compute on detail the shape budget can't render.
UPLOAD_MAX_LONG_SIDE = 720  # HARD SAFETY CAP on input image's long side. Applied at upload
                            # time, BEFORE the engine sees the image. Effective resize ceiling
                            # = min(MAX_RESOLUTION, UPLOAD_MAX_LONG_SIDE). Even on a 102 GB
                            # Blackwell 6000 we hit OOM at native resolutions >1600 because
                            # the bbox-local scorer's per-candidate crop area scales with
                            # max(rx,ry)² ~= (canvas_long_side/8)² — 1600² is 4× the cost
                            # of 720² and pushes K=6144 candidates past the VRAM budget.
                            # 720 keeps every preset comfortably under 20 GB peak. Raise
                            # ONLY if you have measured your GPU's headroom and accept OOM risk.
SEED               = 42     # 0 = time-based
STICKER_MODE       = True   # True if your image has transparency to preserve
ALPHA_THRESHOLD    = 0      # 0 = keep soft alpha. If you see a WHITE HALO around the figure
                            # (common on stickers exported over a white background — the
                            # anti-aliased edge carries white RGB), set 128 to binarize the
                            # silhouette and drop the fringe. Raise to 160-200 if it persists.
CHECKPOINT_EVERY   = 100    # write a recoverable JSON to Drive every N shapes
                            # (survives a mid-run disconnect). 0 = off.
# NOTE: lock_alpha is NOT exposed as a user knob — it's a HARD SYSTEM CONSTRAINT.
# The Forza injector writes alpha=255 unconditionally; soft-alpha JSONs would render
# one way in this notebook's preview and another way in-game. run_gpu raises ValueError
# if cfg.lock_alpha is False. Hardcoded True below in the cfg instantiation.
POLISH_FREEZE_GEOMETRY = True   # True (PRODUCTION DEFAULT): joint_polish refines
                                     # colors only, leaving shape geometry bit-identical to greedy.
                                     # Prevents the inflate/collapse exploits Adam was finding on
                                     # sparkle content (verified at 1000 shapes: 1000/1000 geom
                                     # frozen, engine↔upstream parity 0.000). Set False for the
                                     # legacy joint-geom-and-color polish (escape hatch for future
                                     # structured-refinement experiments).

# --- Gradient-refinement knobs (cheap on memory) ---
GRAD_STARTS        = 16     # multi-start diversity → escapes poor local optima
GRAD_STEPS         = 50     # per-shape convergence depth
GRAD_LR            = 2.0    # Adam learning rate

# Joint-polish: after greedy, gradient-optimize ALL shapes at once vs the full target —
# breaks greedy myopia (early shapes get un-stuck). Fewer shapes benefit from MORE steps
# (each shape carries more responsibility). Watch the printed loss; stop raising when it
# plateaus. 0 = off.
JOINT_POLISH_STEPS = 0

# --- Hill-climb knobs (only used if REFINE_MODE == "hillclimb") ---
MUTATION_ROUNDS    = 16
MUTATIONS_PER_ROUND = 64

print(f"knobs set: shapes={NUM_SHAPES}, samples={RANDOM_SAMPLES}, max_res={MAX_RESOLUTION}, "
      f"edge={EDGE_STRENGTH}, joint_polish={JOINT_POLISH_STEPS}")
print("Next: run the Resolution Planner to validate MAX_RESOLUTION against your VRAM.")


## 8. Resolution Planner — **decide MAX_RESOLUTION**

Shows the detail/VRAM/time tradeoff for your specific image + GPU, recommends the best fit, and **hard-stops if your chosen resolution would OOM**. Re-run the Knobs cell then this one until it says *Proceed to Run*.

In [ ]:
# --- Resolution Planner (decide MAX_RESOLUTION here) ---
# Your sources are high-res PSD exports. Downscaling to fit VRAM throws away detail, so
# pick the LARGEST MAX_RESOLUTION that fits comfortably. This cell shows the tradeoff and
# HARD-STOPS if your current MAX_RESOLUTION would OOM — so you never launch a doomed run.
import torch

_sw, _sh = Image.open(io.BytesIO(SOURCE_IMAGE_BYTES)).size
_long = max(_sw, _sh)
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    _vram_total = _props.total_memory / 1e9
    _vram_free = (_props.total_memory - torch.cuda.memory_allocated()) / 1e9
    _gpu = _props.name
else:
    _vram_total = _vram_free = 0.0
    _gpu = "CPU (no CUDA)"

# Helper used by the probe table AND the hard breakpoint below. MUST stay in
# sync with _load_image_bytes in the Run cell: the final canvas dims it returns
# are exactly what gets handed to run_gpu, which is what the VRAM formula needs.
# Math: effective_max = min(max_resolution, upload_cap); reserve padding budget
# within that (PAD_FACTOR = 1 + 2*BUFFER_FRAC), scale source down to fit the
# reserved input cap, then add pad_px to get the canvas the engine sees.
_BUFFER_FRAC = 0.08
_PAD_FACTOR = 1.0 + 2.0 * _BUFFER_FRAC
def _capped_padded_dims(sw, sh, max_resolution, upload_cap):
    eff_max = max_resolution if upload_cap <= 0 else min(max_resolution, upload_cap)
    eff_in_max = max(1, int(eff_max / _PAD_FACTOR))
    long = max(sw, sh)
    scale = min(1.0, eff_in_max / long)
    iw, ih = max(1, int(sw * scale)), max(1, int(sh * scale))
    pad_px = max(8, int(round(max(iw, ih) * _BUFFER_FRAC)))
    return iw + 2 * pad_px, ih + 2 * pad_px, scale < 1.0 or eff_max < max_resolution

_base_pw, _base_ph, _ = _capped_padded_dims(_sw, _sh, 1200, UPLOAD_MAX_LONG_SIDE)
_base_px = _base_pw * _base_ph

_bbox_on = BBOX_LOCAL and REFINE_MODE == "gradient" and SHAPE_TYPES == ["rotated_ellipse"]
# Effective batch size for memory: bbox scores all RANDOM_SAMPLES ellipses at once; the
# full-canvas multi-kind path scores each shape type's RANDOM_SAMPLES//n_kinds batch
# sequentially, so peak is set by the per-kind batch, not the total.
_eff_k = RANDOM_SAMPLES if _bbox_on else max(1, RANDOM_SAMPLES // max(1, len(SHAPE_TYPES)))
# Peak holds several simultaneous (K, footprint, 3) float32 intermediates. The bbox path
# (gather crops + per-alpha diff + its square) peaks higher than the full-canvas SSE-
# decomposition, so it needs a larger multiplier. These are calibrated against observed peaks.
_SAFETY = 5.5 if _bbox_on else 3.5
print(f"GPU: {_gpu}   VRAM {_vram_total:.0f} GB total, ~{_vram_free:.0f} GB free")
print(f"Source: {_sw}x{_sh} (long side {_long}px)   STICKER_MODE={STICKER_MODE}   "
      f"RANDOM_SAMPLES={RANDOM_SAMPLES}   bbox_local={_bbox_on}")
print(f"Upload cap: {UPLOAD_MAX_LONG_SIDE}px final padded long side (set UPLOAD_MAX_LONG_SIDE=0 to disable).\n")
print(f"{'MAX_RES':>8} | {'final canvas':>13} | {'downscale':>9} | {'peak VRAM':>9} | {'time vs1200':>11} | fit")
print("-" * 76)

_rec = 1200
for _mr in (1200, 1600, 2000, 2400, 3000, 4000, 5000):
    # Final canvas dims = exactly what the engine sees (cap + reserve-for-pad + pad-added).
    _pw, _ph, _capped = _capped_padded_dims(_sw, _sh, _mr, UPLOAD_MAX_LONG_SIDE)
    _px = _pw * _ph
    # bbox-local scores each candidate over a crop (~max-radius sized), not the full canvas,
    # so the per-candidate memory footprint is the crop area, not _px.
    if _bbox_on:
        _crop_e = min(256, max(_pw, _ph) // 8)
        _foot = min((2 * _crop_e + 1) ** 2, _px)
    else:
        _foot = _px
    _peak = _eff_k * _foot * 3 * 4 * _SAFETY / 1e9
    _ratio = _long / max(_pw, _ph)
    _fits = (_vram_free <= 0) or (_peak < _vram_free * 0.85)
    _is_upscale = _mr > _long and not _capped
    if _fits and not _is_upscale:
        _rec = _mr
    _flag = ("OK " if _fits else "OOM") + (" (capped)" if _capped else (" (>=source)" if _is_upscale else ""))
    print(f"{_mr:>8} | {_pw:>5}x{_ph:<7} | {_ratio:>7.1f}x | {_peak:>7.1f}GB | {_px/_base_px:>9.1f}x | {_flag}")
print("-" * 76)
# Detail saturates at the point where the smallest shapes map to real pixels. Past this,
# higher resolution chases detail the shape budget can't reproduce — pure wasted compute.
# Rule of thumb from the geometrize/primitive family: matched long side ~ sqrt(N) * 28.
_matched = int((NUM_SHAPES ** 0.5) * 28)
print(f"Detail saturation for {NUM_SHAPES} shapes: ~{_matched}px long side. Beyond this,")
print(f"  higher MAX_RESOLUTION mostly costs time (raise NUM_SHAPES to exploit a bigger canvas).")
print(f"VRAM-fit recommendation (largest that fits, no upscaling): MAX_RESOLUTION = {_rec}")
print(f"Quality-matched suggestion: MAX_RESOLUTION ~= min({_rec}, {_matched}) = {min(_rec, _matched)}")
print(f"You currently set: MAX_RESOLUTION = {MAX_RESOLUTION}")

# --- Hard breakpoint ---
_pw, _ph, _ = _capped_padded_dims(_sw, _sh, MAX_RESOLUTION, UPLOAD_MAX_LONG_SIDE)
if _bbox_on:
    _crop_e = min(256, max(_pw, _ph) // 8)
    _foot = min((2 * _crop_e + 1) ** 2, _pw * _ph)
else:
    _foot = _pw * _ph
_peak = _eff_k * _foot * 3 * 4 * _SAFETY / 1e9
if _vram_free > 0 and _peak > _vram_free * 0.85:
    raise SystemExit(
        f"\nSTOP: MAX_RESOLUTION={MAX_RESOLUTION} needs ~{_peak:.0f} GB but only ~{_vram_free:.0f} GB "
        f"is free.\n  Fix: set MAX_RESOLUTION={_rec} (or lower RANDOM_SAMPLES) in the Knobs cell, "
        f"re-run it, then re-run this planner."
    )
print(f"\nOK: MAX_RESOLUTION={MAX_RESOLUTION} -> final canvas {_pw}x{_ph}, peak ~{_peak:.0f} GB. Proceed to Run.")
if _long > MAX_RESOLUTION:
    print(f"Detail note: downscaling {_long/MAX_RESOLUTION:.1f}x — features finer than "
          f"~{_long/MAX_RESOLUTION:.0f}px in the source won't survive. Bump MAX_RESOLUTION if you can afford it.")


## 9. Run

Saves the final JSON + PNG to Drive the moment it finishes (and checkpoints every `CHECKPOINT_EVERY` shapes). Progress prints every 50 shapes.

In [ ]:
# --- Run ---
def _load_image_bytes(name, raw, max_resolution, sticker, upload_cap=720):
    img = Image.open(io.BytesIO(raw))
    has_alpha = img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info)
    alpha_mask = None
    if sticker:
        if not has_alpha:
            raise ValueError(f"STICKER_MODE=True but {name} has no alpha (mode={img.mode!r})")
        rgba = np.asarray(img.convert("RGBA"), dtype=np.uint8)
        rgb_img = Image.fromarray(rgba[:, :, :3])  # mode inferred from HxWx3 shape
        alpha_mask = rgba[:, :, 3].copy()
        img = rgb_img
    elif has_alpha:
        rgba = img.convert("RGBA")
        bg = Image.new("RGB", rgba.size, (255, 255, 255))
        bg.paste(rgba, mask=rgba.split()[3])
        img = bg
    else:
        img = img.convert("RGB")
    # Apply the VRAM safety cap BEFORE the preset's max_resolution check. The bbox-local
    # scorer's peak VRAM scales as K * (canvas_long_side / 8)^2 * 3 channels — quadratic in
    # the long side. Even a 102 GB Blackwell 6000 OOMs above ~1600 long-side at K=6144;
    # 720 keeps every production preset comfortably under 20 GB peak. Effective ceiling =
    # min(preset's max_resolution, upload_cap). Setting upload_cap=0 disables the cap.
    effective_max = max_resolution if upload_cap <= 0 else min(max_resolution, upload_cap)
    # The cap is meant as a ceiling on the FINAL PADDED canvas (that's what hits VRAM),
    # not on the input image. Padding below adds ~2*BUFFER_FRAC*long pixels on top of
    # the input. Reserve that budget within the cap so the post-padding canvas fits.
    # Without this, a 720 cap with 8% padding leaked the final canvas up to ~836 px
    # (720 + 2*58) and OOM'd a 102 GB GPU at ~99 GB peak vs the probe's predicted 65 GB.
    BUFFER_FRAC = 0.08
    PAD_FACTOR = 1.0 + 2.0 * BUFFER_FRAC
    effective_input_max = max(1, int(effective_max / PAD_FACTOR))
    original_long = max(img.size)
    if original_long > effective_input_max:
        if effective_input_max < max_resolution:
            print(f"[upload cap] long side {original_long} > {effective_input_max} "
                  f"(input reserved under {effective_max}px final cap; "
                  f"preset max_resolution={max_resolution}). Auto-resizing input to {effective_input_max}.")
    if max(img.size) > effective_input_max:
        scale = effective_input_max / max(img.size)
        new_size = (max(1, int(img.size[0] * scale)), max(1, int(img.size[1] * scale)))
        img = img.resize(new_size, Image.LANCZOS)
        if alpha_mask is not None:
            am_img = Image.fromarray(alpha_mask).resize(new_size, Image.LANCZOS)  # mode inferred
            alpha_mask = np.asarray(am_img, dtype=np.uint8)

    # Edge-buffer padding (matches upstream forza_abyss_painter/shapegen/worker.py). FH6's vinyl renderer
    # treats shapes whose extents touch the canvas edge as unbounded, producing large
    # smears and corner artifacts after injection. Pad the source ~8% per side so even
    # shapes that land on the outermost rows/cols of the content stay several pixels
    # inside the actual canvas edge.
    pad_px = max(8, int(round(max(img.size) * BUFFER_FRAC)))
    new_w = img.size[0] + 2 * pad_px
    new_h = img.size[1] + 2 * pad_px
    if sticker:
        buffered = Image.new("RGB", (new_w, new_h), (0, 0, 0))
        buffered.paste(img, (pad_px, pad_px))
        img = buffered
        if alpha_mask is not None:
            padded_alpha = np.zeros((new_h, new_w), dtype=np.uint8)
            src_h, src_w = alpha_mask.shape[:2]
            padded_alpha[pad_px:pad_px + src_h, pad_px:pad_px + src_w] = alpha_mask
            alpha_mask = padded_alpha
    else:
        # Pad with the SOURCE's mean color, not white. The engine inits the opaque canvas
        # to mean(target) (engine.py:359), so a (255,255,255) margin creates ~33% of the
        # padded canvas as a phantom solid-white region that greedy + polish dutifully
        # spend ~100+ shapes painting before reaching the actual content — pure shape-
        # budget waste. Filling the margin with the source's mean color makes the padded
        # region match the canvas init exactly: per-pixel loss = 0 in the padding, so
        # greedy never places candidates there and the full shape budget goes into the
        # content. (Sticker mode is unaffected; its alpha-mask gate already rejects
        # shapes that don't overlap the silhouette, so its (0,0,0) pad never gets shapes.)
        src_arr = np.asarray(img, dtype=np.uint8)
        src_mean = tuple(int(c) for c in src_arr.reshape(-1, 3).mean(axis=0).round().clip(0, 255))
        buffered = Image.new("RGB", (new_w, new_h), src_mean)
        buffered.paste(img, (pad_px, pad_px))
        img = buffered
    return np.asarray(img, dtype=np.uint8), alpha_mask


target_rgb, alpha_mask = _load_image_bytes(
    SOURCE_IMAGE_NAME, SOURCE_IMAGE_BYTES, MAX_RESOLUTION, STICKER_MODE,
    upload_cap=UPLOAD_MAX_LONG_SIDE,
)
h, w = target_rgb.shape[:2]
mode_tag = "sticker" if STICKER_MODE else "opaque"
print(f"target: {w}x{h} ({mode_tag} mode), refine={REFINE_MODE}, alpha_mask={'yes' if alpha_mask is not None else 'no'}")

cfg = GPUConfig(
    num_shapes=NUM_SHAPES,
    random_samples=RANDOM_SAMPLES,
    seed=SEED,
    refine_mode=REFINE_MODE,
    shape_types=SHAPE_TYPES,
    alpha_levels=ALPHA_LEVELS,
    edge_strength=EDGE_STRENGTH,
    posterize_levels=POSTERIZE_LEVELS,
    bbox_local=BBOX_LOCAL,
    joint_polish_steps=JOINT_POLISH_STEPS,
    alpha_threshold=ALPHA_THRESHOLD,
    grad_starts=GRAD_STARTS,
    grad_steps=GRAD_STEPS,
    grad_lr=GRAD_LR,
    mutation_rounds=MUTATION_ROUNDS,
    mutations_per_round=MUTATIONS_PER_ROUND,
    lock_alpha=True,   # SYSTEM CONSTRAINT — see notebook comment above. Not user-tunable.
    polish_freeze_geometry=POLISH_FREEZE_GEOMETRY,
)
import json
from pathlib import Path

stem = Path(SOURCE_IMAGE_NAME).stem
out_dir = Path(OUTPUT_DIR) / stem
out_dir.mkdir(parents=True, exist_ok=True)
mode_str = f"gpu_colab({mode_tag},shapes={cfg.num_shapes},samples={cfg.random_samples})"

def _doc(shapes):
    # MUST stay aligned with FD6Document.to_dict() in forza_abyss_painter/io/json_schema.py — every field
    # the CPU engine writes, the GPU/notebook must also write. Dropping sticker_mode was
    # the cause of the "looks fine in the engine render, looks wrong in the ForzaAbyssPainter.exe
    # preview/inject" mismatch: the engine optimizes shape colors against a grey substrate
    # when sticker mode is on, and downstream tools read sticker_mode from the JSON to pick
    # the matching backdrop. Without that flag the desktop app paints on white and the
    # grey-substrate-optimized colors look bad.
    return {
        "format": "fd6.shapes", "version": 1,
        "source_image": SOURCE_IMAGE_NAME, "image_size": [w, h],
        "shape_count": len(shapes),
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "profile": mode_str,
        "sticker_mode": bool(STICKER_MODE),
        "shapes": shapes,
    }

# Output naming convention: include NUM_SHAPES (the shape budget) in every filename so
# users can identify quality level at a glance — eg "my_image_3000.json" is a high-detail
# render vs "my_image_400.json" for a lineart-density render. Budget is the planned shape
# count even if the final actual count is slightly lower (sticker constraints can exhaust).
_BUDGET_TAG = str(NUM_SHAPES)

# Checkpoint to Drive during the run so a session reset mid-run still leaves a recoverable
# JSON. Writes greedy progress every CHECKPOINT_EVERY shapes.
def _checkpoint(shape_idx, shapes_so_far):
    p = out_dir / f"{stem}_{_BUDGET_TAG}_ckpt_{shape_idx}.json"
    p.write_text(json.dumps(_doc(shapes_so_far)))

t0 = time.perf_counter()
shapes_json, final_canvas = run_gpu(
    target_rgb, cfg, alpha_mask=alpha_mask, progress_every=50,
    checkpoint_cb=_checkpoint, checkpoint_every=CHECKPOINT_EVERY,
)
elapsed = time.perf_counter() - t0

# Persist FINAL outputs to Drive IMMEDIATELY — before any variable reset / disconnect can
# lose them. This is the whole point of the Drive version. Filenames include the shape
# budget (NUM_SHAPES) so 400/700/1000/3000-shape renders are distinguishable at a glance.
json_path = out_dir / f"{stem}_{_BUDGET_TAG}.json"
png_path = out_dir / f"{stem}_{_BUDGET_TAG}_render.png"
json_path.write_text(json.dumps(_doc(shapes_json), indent=2))
Image.fromarray(final_canvas, "RGB").save(png_path)

print(f"\ndone: {len(shapes_json)} shapes in {elapsed:.2f}s ({elapsed/max(1,len(shapes_json))*1000:.1f} ms/shape)")
print(f"SAVED -> {json_path}")
print(f"SAVED -> {png_path}")
if len(shapes_json) < NUM_SHAPES:
    print(f"WARNING: only {len(shapes_json)} of {NUM_SHAPES} shapes committed (sticker constraint exhausted attempts)")


## 10. View result

In [ ]:
# --- Display source vs render ---
import matplotlib.pyplot as plt

_src_img = Image.open(io.BytesIO(SOURCE_IMAGE_BYTES))
if _src_img.mode != "RGB":
    _src_for_show = _src_img.convert("RGBA") if STICKER_MODE else _src_img.convert("RGB")
else:
    _src_for_show = _src_img

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(_src_for_show)
axes[0].set_title(f"source ({SOURCE_IMAGE_NAME})")
axes[0].axis("off")
axes[1].imshow(final_canvas)
axes[1].set_title(f"Forza Abyss Painter render ({len(shapes_json)} shapes)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 11. (Optional) browser download

Your files are already on Drive. This just pushes a local copy to your browser too — safe to skip, and it won't matter if the session later resets.

In [ ]:
# --- (Optional) also download to your browser ---
# Outputs are ALREADY saved to Drive by the Run cell. This just additionally pushes the
# final files to your browser's downloads if you want a local copy. Safe to skip.
try:
    from google.colab import files
    files.download(str(json_path))
    files.download(str(png_path))
except Exception as _e:
    print(f"Browser download skipped ({_e}). Your files are on Drive at: {out_dir}")


<div align="center" style="margin-top: 32px; padding: 16px; border-top: 1px solid #3a2555; color: #6a6a6a; font-size: 12px;">
  <p style="margin: 0;">made with <strong style="color: #d94f90;">Forza Abyss Painter</strong></p>
  <p style="margin: 4px 0 0 0;"><a href="https://github.com/whykusanagi/forza-abyss-painter" style="color: #8b5cf6; text-decoration: none;">github.com/whykusanagi/forza-abyss-painter</a> · MIT</p>
</div>
